# sources

> Add concepts to the concept graph from any text, from a vault document and from vedicreader.

This module adds concepts to the `learning` shelf. A model finds the core ideas in a text, and code merges them with the concepts on the shelf.
The vedicreader import reads the repo files and needs no model.
The tutor then asks about the concepts that this module adds.

In [ ]:
#| default_exp sources

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

These are the imports of the module.

In [ ]:
#| export
import json, os, re, shutil
from pathlib import Path
from fastcore.utils import *
from abhyasa.store import Learning, SHELF, slugify, _union, clean_group, clean_subject, readable_name, usable_name, under
from abhyasa.settings import val, source_root_why, NO_FOLDER
from abhyasa.llm import ask_json, for_task
from abhyasa.prompts import register, prompt

## Extraction from text

The model lists the core ideas of a text. Each idea has a true, self-contained statement and its prereqs.
Code does the merge. When a slug or an alias matches a concept on the shelf, it is the same concept.
The lists of that concept get the new items, and its learner state does not change.

`EXTRACT_SP` and `extract_prompt` ask the model for the concepts. The model also gives the section, one quote, the figure, the ways that suit the idea, and the people and dates. The model also gives a group path. The prompt lists the groups of the subject, so the tree does not split into many groups with almost the same name. `add_group` adds the new groups of a reply for the next window. `quoted` puts each name on one line in JSON quotes, so a name cannot add a line to the prompt. The prompt says when an idea gets `order`, `demo` or `code`, because a walk gives an idea with one of these ways a check of that kind. `EXTRACT_SP` is a registered prompt, so you can change it on the Settings page.

In [ ]:
#| export
EXTRACT_SP = register('extract', 'Concept finder', 'How the tutor finds the ideas in a text that you add.', """You find the core ideas in a text, for a learner who will be tested on them.
- A concept is one idea a learner can be asked about: a definition, a rule, a method, a result, a distinction.
- `statement` is true and self-contained: it makes sense without the text. One to three sentences. Maths in $...$.
- Skip trivia, examples and names that are not ideas. Prefer 4-15 concepts for a page or two.
- `prereqs` names other concepts (from your list or from the known list) that a learner must know first.
- List the concepts in the order of the text.
- `section` is the title of the part of the text that holds the idea.
- `quote` is one short sentence of the text, copied exactly.
- `figure` is the link of one image of the text that shows the idea, copied exactly. Use "" when there is none.
- `ways` names the ways that suit the idea. Use only names from the list in the request.
- Add `order` when the idea is a process or a sequence of steps. Add `demo` when a change to one value changes the result. Add `code` when a short Python program with only the standard library can show the idea. Do not add these ways to other ideas.
- `moves` names the things that change in the idea, for example "weights". Use [] when nothing changes.
- `when` is the year of the idea, or null. `who` names the people of the idea.
- `group` is the place of the idea under the subject: one or two short names, from wide to narrow, for example ["nakshatras"]. Do not use the subject as a group. Use a group from the list of groups when one fits. Make a new group only when no group fits. Use [] when you are not sure.
- When the request has no `Subject` line, `subject` is the subject of the idea: one name from the list of subjects when one fits, or a short new name of one to three words, for example "transformers". Use one subject for all ideas of the text, unless an idea clearly belongs to another subject in the list. When the request has a `Subject` line, do not give `subject`.
- `name` is a readable name of the idea, in words. Do not use a slug with hyphens as a name.""")

def quoted(s) -> str:
    "`s` on one line, in JSON quotes, so a name with a line break or a leading `- ` cannot add a line to a prompt."
    return json.dumps(' '.join(str(s or '').split()), ensure_ascii=False)

def group_line(groups) -> str:
    "The line of a prompt that lists the group paths of a subject, or ''."
    return f"Groups of this subject now: {'; '.join(' › '.join(g) for g in groups[:200])}\n" if groups else ''

def add_group(gs:list, g:list) -> list:
    "Add the path `g` and its parents to the paths `gs` when they are not there. Case does not count. Return `gs`."
    have = {tuple(x.lower() for x in p) for p in gs}
    for i in range(1, len(g) + 1):
        if tuple(x.lower() for x in g[:i]) not in have: gs.append(g[:i]); have.add(tuple(x.lower() for x in g[:i]))
    return gs

def extract_prompt(text, subject='', known=(), groups=(), subjects=(), lead=''):
    from abhyasa.method import METHODS
    kn = ', '.join(map(quoted, known[:150]))
    return ((f"Subject: {quoted(subject)}\n" if subject else '') + (f"Subjects the learner has: {'; '.join(map(quoted, subjects[:60]))}\n" if subjects and not subject else '')
            + (f"Subject of this text so far: {quoted(lead)}\n" if lead and not subject else '')
            + (f"Concepts the learner already has: {kn}\n" if kn else '') + group_line(groups)
            + f"Ways: {', '.join(METHODS)}\n\nText:\n{text}\n\n"
            'Return JSON: {"concepts": [{"name": str, "statement": str, "aliases": [str], "prereqs": [str], "section": str, "quote": str, '
            '"figure": str, "ways": [str], "moves": [str], "when": int|null, "who": [str], "group": [str]' + ('' if subject else ', "subject": str') + '}]}')

`_index` finds the slug for a name or an alias. `_windows` cuts a long text at paragraph breaks. `doc_pages` reads every chunk of a vault document in page order. It adds 1 to the page, because the vault counts pages from 0 and a reader counts from 1.

In [ ]:
#| export
def _index(lr):
    "Map each lower-case name, alias and slug on the shelf to its slug."
    out = {}
    for c in lr.concepts():
        for n in (c.slug, c.name, *(c.aliases or ())): out[n.lower()] = c.slug
    return out

def _windows(text, size=12000):
    "Cut a long text at paragraph breaks into windows of about `size` characters."
    out, cur = [], ''
    for p in re.split(r'\n\s*\n', text):
        if cur and len(cur) + len(p) > size: out.append(cur); cur = ''
        cur += p + '\n\n'
    return [*out, cur] if cur.strip() else out

`doc_pages` returns the chunks of a document. When the main shelf has no such document, it looks on the other shelves, but not on the `learning` shelf of your concepts.

In [ ]:
#| export
def doc_pages(lr:Learning, ref:str, shelf:str='store') -> AttrDict:
    "Every chunk of one vault document in page order, as `(page, text)` with the page a reader sees (from 1)."
    sh = lr.vault if shelf == 'store' else lr.vault.shelf(shelf)
    if (d := sh.doc(ref)) is None and shelf == 'store':   # a notebook or a folder file is on its own shelf: look on each shelf but the concepts
        for name in (s['store'] for s in lr.vault.shelves() if s['store'] not in ('store', SHELF)):
            if (d := (o := lr.vault.shelf(name)).doc(ref)) is not None: sh = o; break
    if d is None: raise ValueError(f'no document in the vault matching {ref!r}')
    did = d['id'].replace("'", "''")
    cs = sorted(sh.store(where=f"doc_id='{did}'", select='content, page, rowid as rowid'), key=lambda c: (-1 if c['page'] is None else c['page'], c['rowid']))
    return AttrDict(doc_id=d['id'], title=d['title'], source=d['source'], pages=[(None if c['page'] is None else c['page'] + 1, c['content']) for c in cs])

`locate` finds a quote in the pages. It folds white space and case, and it looks in all pages, so a window border does not lose a quote.

In [ ]:
#| export
def _fold(s):
    "`s` in lower case with each run of white space made one space, and the place in `s` of each character."
    out, at = [], []
    for i, ch in enumerate(s):
        if ch.isspace():
            if out and out[-1] != ' ': out.append(' '); at.append(i)
        else:
            for x in ch.lower(): out.append(x); at.append(i)
    return ''.join(out), at

def _locate(pages, quote):
    "`(index of the chunk, start, end)` of `quote` in `pages`, or None."
    q = _fold((quote or '').strip())[0]
    if len(q) < 12: return None
    for i, (_, t) in enumerate(pages):
        f, at = _fold(t)
        if (j := f.find(q)) >= 0: return i, at[j], at[j + len(q) - 1] + 1

def locate(pages:list, quote:str):
    "`(page, start, end)` of `quote` in the `(page, text)` list `pages`, with white space and case folded. None when it is not there or too short."
    r = _locate(pages, quote)
    return r and (pages[r[0]][0], r[1], r[2])

`figure_path` keeps a figure only when the file is in the assets folder of the document and is an image.

In [ ]:
#| export
IMG_SUFFIXES = ('.png', '.jpg', '.jpeg', '.gif', '.webp')

def figure_path(lr:Learning, source:str, link:str):
    "The path of the image `link` of the document `source`, from the assets folder. None when the file is not there, is outside the folder of that document or is not an image."
    if not (source and link): return None
    root = lr.vault.assets().resolve()
    for name in (Path(source).stem, source.rsplit('/', 1)[-1].split('?')[0]):   # the folder of a file, or of a link: the vault names a PDF link by its last part
        try: base = lr.vault.assets(name).resolve(); p = (base/str(link)).resolve()
        except (ValueError, OSError): continue
        if base.parent == root and p.suffix.lower() in IMG_SUFFIXES and p.is_file() and p.is_relative_to(base): return str(p.relative_to(root))


`placement` gives the entry of a concept for one document.

In [ ]:
#| export
def placement(c, doc_id:str):
    "The newest provenance entry of the concept `c` with this `doc_id`, or None."
    return next((AttrDict(p) for p in reversed(c.get('provenance') or ()) if p.get('doc_id') == doc_id), None)

`extract` adds the concepts of a text. When it gets pages, it also keeps where each idea is in the text: the order, the section, the page and the passage. `_read` asks the model for each window. `_places` puts the ideas in the order of the text. `from_vault` does the same for one document of the vault.

`math_cut` cuts a text to a length, but never inside a formula. A cut inside `$…$` or `$$…$$` moves back to before the formula, so a cut passage or a cut page never shows half a formula. An escaped `\$` is not a formula mark. A web page can have a `$` that is a price: a cut never moves back more than `MATH_MOST` characters, so a page does not lose its text.

In [ ]:
#| export
_DOLLAR = re.compile(r'\\.|\$\$|\$', re.S)
MATH_MOST = 1000   # a formula is shorter than this: a `$` further back is a price in a web page, not a formula

def math_cut(s:str, n:int) -> str:
    "`s` cut to at most `n` characters, never inside a `$…$` or `$$…$$` formula: a cut in a formula moves back to before it. A `$` more than `MATH_MOST` characters back is not a formula."
    s = s or ''
    if len(s) <= n: return s
    t, at, mark = s[:max(n, 0)], 0, None
    for m in _DOLLAR.finditer(t):
        g = m.group()
        if g[0] == '\\': continue
        if mark is None: mark, at = g, m.start()
        elif g == mark: mark = None
        elif mark == '$': at = m.start() + 1   # `$a$$b$`: one formula ends and the next one starts
    return t[:at] if mark and len(t) - at <= MATH_MOST else t

A cut in an inline or a block formula moves back to before it. A cut after a whole formula, or in text with an escaped `\$`, stays where it is.

In [ ]:
f = r'We scale by $\frac{1}{\sqrt{d_k}}$ here.'
test_eq(math_cut(f, 20), 'We scale by '); test_eq(math_cut(f, len(f) - 1), r'We scale by $\frac{1}{\sqrt{d_k}}$ here')
test_eq(math_cut('A $$x=1$$ and $$y=2$$.', 18), 'A $$x=1$$ and ')
test_eq(math_cut(r'Pay \$5 and \$6 now.', 12), r'Pay \$5 and ')
test_eq(math_cut('$a$$b$ c', 5), '$a$'); test_eq(math_cut('short', 10), 'short')
test_eq(len(math_cut('It costs $5. ' + 'x' * 5000, 3000)), 3000)   # a price far back: the cut stays

In [ ]:
#| export
def _strs(x): return [s for s in x if isinstance(s, str)] if isinstance(x, list) else []

def _doc_source(lr, source):
    "The source path of the vault document that `source` names, or ''."
    if not (source and source.get('doc_id')): return ''
    sh = lr.vault if source.get('shelf', 'store') == 'store' else lr.vault.shelf(source['shelf'])
    return (sh.doc(source['doc_id']) or {}).get('source') or ''

def _read(lr, ask, text, subject, pages):
    "Ask for the concepts of each window of the text, and return `(slug, concept, prereq slugs)` in the order of the replies."
    idx, out, gs = _index(lr), [], lr.groups(subject)
    have, lead = [c['subject'] for c in lr.subjects() if c['subject']], ''
    for w in _windows('\n\n'.join(t for _, t in pages)):
        r = ask_json(for_task(ask, 'extract'), extract_prompt(w, subject, sorted({v for v in idx.values()}), gs, have, lead), prompt('extract'))
        cs = [c for c in (r.get('concepts', []) if isinstance(r, dict) else r) if isinstance(c, dict)]
        for c in cs: c['name'] = readable_name(c.get('name'), c.get('statement'))
        cs = [c for c in cs if usable_name(c['name'])]   # a name such as `x` is not an idea
        slug_of = {}
        for c in cs:
            names = [c['name'], *(c.get('aliases') or ())]
            slug_of[c['name'].lower()] = next((idx[n.lower()] for n in names if n.lower() in idx), slugify(c['name']))
        for c in cs:
            s = slug_of[c['name'].lower()]
            out.append((s, c, [slug_of.get(p.lower()) or idx.get(p.lower()) for p in c.get('prereqs') or ()]))
        if not subject:   # one subject for the text: the most common one that the finder gave for a new idea
            for c in cs: c['subject'] = next((h for h in have if h.lower() == clean_subject(c.get('subject'))), clean_subject(c.get('subject')))
            new = [c['subject'] for c in cs if c['subject'] and slug_of[c['name'].lower()] not in idx.values()]
            lead = lead or (max(set(new), key=new.count) if new else '')
            for c in cs: c['subject'] = c['subject'] or lead
        for c in cs: add_group(gs, clean_group(c.get('group'), subject or c.get('subject')))   # the next window sees the new groups
        for c in cs: idx.update({n.lower(): slug_of[c['name'].lower()] for n in (c['name'], *(c.get('aliases') or ()))})
    return out

def _places(pages, recs):
    "The place of each concept in the text: a list of `(order, located)`. A concept that is not found follows the one before it."
    locs, key, keys = [_locate(pages, c.get('quote')) for _, c, _ in recs], (-1, -1), []
    for l in locs: key = l[:2] if l else key; keys.append(key)
    rank = {i: n for n, i in enumerate(sorted(range(len(recs)), key=keys.__getitem__))}
    return [(rank[i], l) for i, l in enumerate(locs)]

def extract(lr:Learning, ask, text:str='', subject:str='', source:dict=None, pages:list=None) -> L:
    "Add the core ideas of `text` (or of `pages`, a list of `(page, text)`) to the shelf as concepts, merged with the concepts there. Return the slugs that it changed."
    subject, pgs = clean_subject(subject), pages if pages is not None else [(None, text)]
    recs = _read(lr, ask, text, subject, pgs)
    places = _places(pgs, recs) if pages is not None else [(None, None)] * len(recs)
    src = _doc_source(lr, source) if pages is not None else ''
    touched, kept = L(), {}
    for (s, c, pre), (order, loc) in zip(recs, places):
        old, fig = lr.concept(s) or {}, figure_path(lr, src, c.get('figure'))
        moves = _union(old.get('moves'), _strs(c.get('moves')))
        meta = dict(ways=_union(old.get('ways'), _strs(c.get('ways'))), moves=moves, who=_union(old.get('who'), _strs(c.get('who'))),
                    when=c.get('when', old.get('when')), image=fig or old.get('image'), visual=bool(moves))
        subj = subject or old.get('subject') or c.get('subject') or ''
        if subj not in kept: kept[subj] = lr.groups(subj)
        g = clean_group(c.get('group'), subj)
        if g and not old.get('group') and old.get('group_by') not in ('hand', 'undone') and (not kept[subj] or g in kept[subj]):   # with groups, only one of them, spelled the same way
            meta.update(group=g, group_by='placed' if kept[subj] else 'model')   # `placed`: the subject had groups, so the tutor placed it and you can undo it
        prov = [source] if source else []
        if pages is not None and source is not None:
            t = math_cut(pgs[loc[0]][1][loc[1]:loc[2]], val('walks.passage')) if loc else ''
            prov = [{**source, 'order': order, 'section': c.get('section') or '', 'page': pgs[loc[0]][0] if loc else None,
                     'span': [loc[1], loc[1] + len(t)] if loc else None, 'passage': t, 'figure': fig}]
        lr.put(c['name'], c.get('statement', ''), subj, slug=s, aliases=c.get('aliases') or (), prereqs=[p for p in pre if p], provenance=prov, **meta)
        touched.append(s)
    return touched.unique()

def _cut(pages, chars):
    "The `(page, text)` list `pages`, cut to `chars` characters in all; all of it when `chars` is None."
    if chars is None: return pages
    out = []
    for p, t in pages:
        if chars <= 0: break
        out.append((p, math_cut(t, chars))); chars -= len(t)
    return out

def from_vault(lr:Learning, ask, ref:str, subject:str='', shelf:str='store', chars:int=None) -> L:
    "Extract concepts from one document of a shelf of the vault, by doc id, source or title; from its first `chars` characters when `chars` is given."
    d = doc_pages(lr, ref, shelf)
    return extract(lr, ask, subject=subject, source=dict(src='vault', shelf=shelf, doc_id=d.doc_id, title=d.title), pages=_cut(d.pages, chars))

A fake model returns three concepts. One matches `Ranking` on the shelf by its alias `rank`.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Ranking', 'Order items by a score.', 'ml', aliases=['rank'])
fake = Fake({'Return JSON: {"concepts"': dict(concepts=[
    dict(name='Reciprocal rank fusion', statement='Sum 1/(k + rank) over lists.', aliases=['RRF'], prereqs=['rank', 'Ranked list']),
    dict(name='Ranked list', statement='Items in score order.', aliases=[], prereqs=[]),
    dict(name='Ranking', statement='Ordering by score.', aliases=['ordering'], prereqs=[])])})

`extract` returns the changed slugs. The prereqs become slugs, and the old concept gets the new alias and statement.

In [ ]:
test_eq(sorted(extract(lr, fake, 'some text', 'ml', dict(src='test'))), ['ranked-list', 'ranking', 'reciprocal-rank-fusion'])
test_eq(lr.concept('reciprocal-rank-fusion').prereqs, ['ranking', 'ranked-list'])
test_eq(lr.concept('ranking').aliases, ['rank', 'ordering'])
test_eq(lr.concept('ranking').statement, 'Ordering by score.')

`_windows` cuts at paragraph breaks.

In [ ]:
test_eq(len(_windows('a\n\n' * 10, size=5)), 5)

### A group for each new concept

The model also gives each new concept a group path under the subject. The prompt lists the groups that the subject has now, so a new concept goes into one of them when it fits.
Here the subject has a group from the tutor and a group that the learner made by hand.

In [ ]:
gl = temp_learning()
gl.put('Ashvini', 'The first nakshatra.', 'jyotisha', group=['nakshatras'], group_by='model')
gl.put('Magha', 'The tenth nakshatra.', 'jyotisha', group=['Nakshatras', 'Gana'], group_by='hand')
gfake = Fake({'Return JSON: {"concepts"': dict(concepts=[
    dict(name='Bharani', statement='The second nakshatra.', group=['jyotisha', 'nakshatras']),
    dict(name='Magha', statement='The tenth nakshatra, ruled by Ketu.', group=['rasis']),
    dict(name='Ashvini', statement='The first nakshatra.', group=['stars']),
    dict(name='Lagna', statement='The rising sign.', group=['Signs', 'Rising', 'More'])])})
test_eq(sorted(extract(gl, gfake, 'some text', 'jyotisha')), ['ashvini', 'bharani', 'lagna', 'magha'])

The prompt asks for the group and lists the groups of the subject, with their parents.

In [ ]:
p = gfake.calls[0][0]
assert '"group": [str]' in p and 'Groups of this subject now: nakshatras; Nakshatras › Gana' in p, p

A new concept gets its group, cleaned: the subject drops. When the subject has groups, the group must be one of them, spelled the same way; else the concept has no group. A concept that has a group keeps it. A concept that you moved by hand stays.

In [ ]:
test_eq((gl.concept('bharani').group, gl.concept('bharani').group_by), (['nakshatras'], 'placed'))   # the subject has groups: placed by the tutor
test_is(gl.concept('lagna').get('group'), None)   # `Signs` is a new group: the tutor does not make it by itself
test_eq(gl.concept('ashvini').group, ['nakshatras'])
test_eq((gl.concept('magha').group, gl.concept('magha').group_by), (['Nakshatras', 'Gana'], 'hand'))

### A subject and a readable name for each new concept

When the learner gives no subject, the finder gives one for each idea: a subject that exists, or a short new one. All ideas of one text get one subject, unless an idea clearly belongs to another subject that exists. A concept that has a subject keeps it. A name that is empty, or is a slug, becomes words.

In [ ]:
sx = temp_learning(); sx.put('Rope', 'A position code.', 'physics')
srep = dict(concepts=[dict(name='Softmax', statement='Scores to weights.', subject='Transformers'),
    dict(name='attention-heads-and-anaphora-resolution', statement='Heads.', subject='transformers'),
    dict(name='', statement='A rank is a position in a list.'), dict(name='Rope', statement='Again.', subject='ml')])
sxf = Fake({'Return JSON: {"concepts"': srep})
test_eq(sorted(extract(sx, sxf, 'text')), ['a-rank-is-a-position-in', 'attention-heads-and-anaphora-resolution', 'rope', 'softmax'])
test_eq([sx.concept(s).subject for s in ('softmax', 'attention-heads-and-anaphora-resolution', 'a-rank-is-a-position-in', 'rope')], ['transformers', 'transformers', 'transformers', 'physics'])
test_eq(sx.concept('attention-heads-and-anaphora-resolution').name, 'Attention heads and anaphora resolution')
test_eq(sx.concept('a-rank-is-a-position-in').name, 'A rank is a position in')
assert 'Subjects the learner has: "physics"' in sxf.calls[0][0] and '"subject": str' in sxf.calls[0][0]

With a subject from the learner, the finder is not asked for one, and each new concept gets the subject of the learner.

In [ ]:
sy = temp_learning(); syf = Fake({'Return JSON: {"concepts"': srep})
extract(sy, syf, 'text', 'ml')
test_eq({c.subject for c in sy.concepts()}, {'ml'})
assert '"subject": str' not in syf.calls[0][0] and 'Subjects the learner has' not in syf.calls[0][0]

A subject from the learner is cleaned (short, plain, lower case) before it goes in the prompt, in quotes, and on the concepts.

In [ ]:
sz = temp_learning(); szf = Fake({'Return JSON: {"concepts"': srep})
extract(sz, szf, 'text', ' ML  Stuff!\n- evil: x ')
test_eq({c.subject for c in sz.concepts()}, {'ml stuff - evil x'})
assert 'Subject: "ml stuff - evil x"\n' in szf.calls[0][0] and not any(l.startswith('- evil') for l in szf.calls[0][0].splitlines())

## Reading order and where each idea stands

When the text comes as pages, `extract` also keeps the order, the section, the page and the passage of each idea. These tests use a small paper in a temporary vault. The paper data is in `tests/walk_data.py`.

In [ ]:
#| hide
import sys, abhyasa
from pathlib import Path
sys.path.insert(0, str(Path(abhyasa.__file__).parents[1]/'tests'))
from walk_data import *
from abhyasa.prompts import PROMPTS

A fake model names three ideas of the paper. `from_vault` returns them in the order of the text.

In [ ]:
lr = temp_learning(); did = add_paper(lr)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY})
slugs = from_vault(lr, ask, did)
test_eq(list(slugs), ['sequential-computation', 'softmax', 'scaling-by-the-root-of-dk'])

The placement of an idea is in its provenance. The page is the page a reader sees: the vault counts pages from 0, so the stored page 3 is page 4.

In [ ]:
p = placement(lr.concept('softmax'), did)
test_eq((p.order, p.section, p.page), (1, '3.2.1 Scaled Dot-Product Attention', 4))
assert p.passage.startswith('apply a softmax function')
test_eq(p.figure, 'attention/images/fig2.png')

The way fields, the picture and the people go on the concept.

In [ ]:
test_eq(lr.concept('softmax').ways, ['code', 'maths'])
test_eq(lr.concept('softmax').visual, True)
test_eq(lr.concept('softmax').image, 'attention/images/fig2.png')
test_eq(lr.concept('scaling-by-the-root-of-dk').who, ['Vaswani'])

A second document that names Softmax keeps its own placement. The ways of the two documents merge.

In [ ]:
d2 = lr.vault.add([(0, 'Softmax gives weights.')], 'Notes', source='/tmp/notes.md', kind='file')['doc_id']
from_vault(lr, Fake({'Return JSON: {"concepts"': dict(concepts=[dict(name='Softmax', statement='', quote='Softmax gives weights.', section='', ways=['words'])])}), d2)
test_eq(placement(lr.concept('softmax'), did).page, 4)
test_eq(placement(lr.concept('softmax'), d2).page, 1)
test_eq(lr.concept('softmax').ways, ['code', 'maths', 'words'])

A quote that is not in the text gives no passage. A figure file that is not there gives no figure.

In [ ]:
bad = dict(concepts=[dict(name='Masking', statement='The decoder hides later positions.', quote='we mask out all values', figure='images/nope.png', section='3.2.3')])
lr2 = temp_learning(); d3 = add_paper(lr2); from_vault(lr2, Fake({'Return JSON: {"concepts"': bad}), d3)
m = placement(lr2.concept('masking'), d3)
test_eq((m.passage, m.span, m.figure), ('', None, None))

A long paper has 15 pages. The model sees the paper in windows, and the last idea still gets its page.

In [ ]:
long = [(i, f'{i + 1} Section {i + 1}\n\n' + f'Idea {i + 1} is stated here. ' * 140) for i in range(15)]
lr3 = temp_learning(); d4 = lr3.vault.add(long, 'Long paper', source='/tmp/long.pdf', kind='pdf')['doc_id']
def per_window(p, sp):
    ns = sorted({int(n) for n in re.findall(r'Idea (\d+) is stated', p)})
    return json.dumps(dict(concepts=[dict(name=f'Idea {n}', statement=f'Idea {n}.', quote=f'Idea {n} is stated here.', section=f'{n} Section {n}') for n in ns]))
from_vault(lr3, per_window, d4)
test_eq(placement(lr3.concept('idea-15'), d4).page, 15)

`locate` folds white space and case. It refuses a quote that is too short.

In [ ]:
pg = [(1, 'One two.\n\nThree   FOUR five six.'), (2, 'Seven eight nine ten.')]
test_eq(locate(pg, 'three four FIVE six'), (1, 10, 31))
test_eq(locate(pg, 'seven eight nine'), (2, 0, 16))
test_is(locate(pg, 'three'), None)
test_is(locate(pg, 'nothing of this is there'), None)

`figure_path` keeps a figure only when the file is there, stays in the assets folder of that one document and has an image suffix. The folder of a PDF link is named by the last part of the link, such as `1706.03762`, which `Path.stem` would cut to `1706`.


In [ ]:
test_eq(figure_path(lr, '/tmp/attention.pdf', 'images/fig2.png'), 'attention/images/fig2.png')
test_is(figure_path(lr, '/tmp/attention.pdf', '../../../etc/hosts'), None)
test_is(figure_path(lr, '/tmp/attention.pdf', '/etc/hosts'), None)
(lr.vault.assets('attention')/'images'/'x.txt').write_text('x')
test_is(figure_path(lr, '/tmp/attention.pdf', 'images/x.txt'), None)
(lr.vault.assets('1706.03762')/'images'/'page3_1.png').parent.mkdir(parents=True, exist_ok=True); (lr.vault.assets('1706.03762')/'images'/'page3_1.png').write_bytes(b'x')
test_eq(figure_path(lr, 'https://arxiv.org/pdf/1706.03762', 'images/page3_1.png'), '1706.03762/images/page3_1.png')   # a link, not a file
(lr.vault.assets('other')/'images').mkdir(parents=True, exist_ok=True); (lr.vault.assets('other')/'images'/'o.png').write_bytes(b'x')
test_is(figure_path(lr, '/tmp/attention.pdf', '../other/images/o.png'), None)   # the image of another document
test_is(figure_path(lr, '/tmp/..', 'assets/other/images/o.png'), None)          # a source with no name of its own
test_is(figure_path(lr, '/', 'other/images/o.png'), None)

## Sort your concepts into groups

An old vault has no groups. The Concepts page has the action "Sort my concepts". It asks the model to place the concepts of each subject in a tree of at most two levels.
The model reads about `lists.sort_batch` concepts in each request. Each request lists the groups so far, so the batches use the same groups.
`propose_groups` saves nothing: the page shows the tree, and `apply_groups` saves it when the learner accepts it. Neither one changes a concept that the learner moved by hand.

`SORT_SP` and `sort_prompt` ask for the place of each concept in one batch.

In [ ]:
#| export
SORT_SP = register('concepts.sort', 'Concept sorter', 'How the tutor sorts your concepts into groups on the Concepts page.', """You sort the concepts of one subject into a small tree of groups, so that a learner can find them again.
- `group` is one or two short names, from wide to narrow, for example ["nakshatras"] or ["grammar", "sandhi"]. Do not use the subject as a group.
- Use a group from the list of groups first. Make a new group only when no group in the list fits. Do not rename a group in the list.
- A good group has three concepts or more. Do not make a group for one concept when a wider group fits.
- Use plain names of one to three words, in the words of the subject. Spell a group the same way each time.
- Give a group to each concept in the list. Do not leave a concept out. Copy its slug exactly.
- When no group fits a concept, use the group "Other". Use "Other" only as the last resort, for a few concepts.
- When the request says "Sort all these concepts again", the list has only the groups that the learner made. Keep these groups, with the same names. Make new groups for the other concepts.
- A concept can have no subject. Then also give `subject`: one name from the list of subjects when one fits, or a short new name of one to three words. Give the group of the concept under that subject.""")

def sort_prompt(subject:str, cs, groups=(), subjects=(), again:bool=False, closed:bool=False, every:bool=False) -> str:
    "The request for one batch: the subject (or the subjects that exist, for concepts with no subject), the groups so far, and one line for each concept. `again` asks for the concepts that got no place. `closed` allows only a group from the list: any other answer is []. `every` says that the sort starts again, and that the groups in the list are the groups that the learner made."
    item = lambda c: f"- {c.slug}: {quoted(c.name)}. {quoted(str(c.get('statement') or '')[:160])}"
    head = f"Subject: {quoted(subject)}\n" if subject else "Subject: none. These concepts have no subject.\nSubjects now: " + ('; '.join(map(quoted, subjects[:60])) or 'none') + '\n'
    return (head + (group_line(groups) or 'Groups of this subject now: none\n')
            + ('\nSort all these concepts again. The groups in the list are the groups that the learner made.\n' if every else '')
            + ('\nThese concepts got no place the first time. Give a group to each one now.\n' if again else '')
            + ('\nUse only a group from the list, spelled the same way. Do not make a new group and do not use "Other". Use [] for a concept that fits no group in the list.\n' if closed else '')
            + '\nConcepts:\n' + '\n'.join(map(item, cs))
            + '\n\nReturn JSON: {"places": [{"slug": str, "group": [str]' + ('' if subject else ', "subject": str') + '}]}')

`propose_groups` asks for each subject in batches and returns `{slug: path}`. `apply_groups` saves the paths that change. The sort uses the model of the concept finder.

In [ ]:
#| export
class Places(dict):
    "The result of a sort: `{slug: group}`, with `.subjects` (`{slug: subject}` for concepts that had none) and `.left` (the slugs that got no place)."
    def __init__(self, *a, subjects=None, left=None): super().__init__(*a); self.subjects, self.left = subjects or {}, left or []

def propose_groups(lr:Learning, ask, subject:str=None, batch:int=None, only=None, closed:bool=False, every:bool=False, free=()) -> Places:
    "Ask the model for the group of each concept that has no group (and that you did not keep out of the groups by hand), `batch` concepts at a time for each subject; ask again once for the concepts that got no place. Concepts with no subject also get one. A concept with a group never moves. `only` limits it to these slugs. With `closed`, a group must be in the list of the subject, spelled the same way, and it asks one time for each batch; any other answer leaves the concept out. With `every`, it sorts every concept again, except a concept that you moved by hand and that is not in `free`, and the list of groups is only the hand groups. Save nothing."
    batch, out, by = batch or val('lists.sort_batch'), Places(), {}
    have = [c['subject'] for c in lr.subjects() if c['subject']]
    for c in (lr.concepts(subject).filter(lambda c: c.get('group_by') != 'hand' or c.slug in free) if every else lr.unsorted(subject)):
        if only is None or c.slug in only: by.setdefault(c.subject or '', []).append(c)
    def place(s, part, gs, again=False):
        "Ask for the places of `part`; keep each one with a group. Return the slugs of `part` that got none."
        r = ask_json(for_task(ask, 'extract'), sort_prompt(s, part.values(), gs, have, again, closed, every), prompt('concepts.sort'))
        for p in (r.get('places') if isinstance(r, dict) else None) or []:
            if not (isinstance(p, dict) and p.get('slug') in part): continue
            sj = s or next((h for h in have if h.lower() == clean_subject(p.get('subject'))), clean_subject(p.get('subject')))
            if (g := clean_group(p.get('group'), sj)) and (not closed or g in gs):
                out[p['slug']] = g; add_group(gs, g)
                if sj and not s: out.subjects[p['slug']] = sj
        return [k for k in part if k not in out]
    for s, cs in sorted(by.items()):
        gs = [] if every else lr.groups(s)
        if every:
            for g in lr.hand_groups(s): add_group(gs, g)
        for i in range(0, len(cs), batch):
            part = {c.slug: c for c in cs[i:i + batch]}
            if (miss := place(s, part, gs)) and not closed: out.left += place(s, {k: part[k] for k in miss}, gs, True)
            elif miss: out.left += miss
    return out

def apply_groups(lr:Learning, places:dict, every:bool=False, free=(), unmark=()) -> list:
    "Save the paths of `places` as a saved sort, and keep the group list of each subject. A concept with no subject gets its proposed subject first. Return the slugs that moved. A concept that has a group, that you moved by hand, or that is gone, stays. With `every`, a concept with a group moves too, and so does a concept that you moved by hand when it is in `free`; the group list of each subject becomes the new groups and the hand groups. `unmark` (`(subject, path)`) takes the hand mark off those groups first. It is one step: when a write fails, it writes all back and raises."
    with lr._lock:
        cs = [c for c in map(lr.concept, places) if c]
        snap = lr.snapshot([c.slug for c in cs], {*(c.subject or '' for c in cs), *getattr(places, 'subjects', {}).values(), *(s for s, _ in unmark)})
        try: return _apply(lr, places, every, free, unmark)
        except Exception: lr.restore(snap); raise

def _apply(lr, places, every, free, unmark):
    for s, g in unmark: lr.unmark_groups(s, [g])   # the groups that you unticked: the tutor sorts them now
    moved, subjects = [], {}
    for slug, g in places.items():
        c = lr.concept(slug)
        if c is None or (c.get('group_by') == 'hand' and slug not in free) or (c.get('group') and not every): continue   # a concept with a group is sorted: it stays
        if c.get('group_by') == 'hand': c = lr.update(slug, group_by=None)   # you let the tutor sort it again
        if (sj := getattr(places, 'subjects', {}).get(slug)) and not c.subject: c = lr.set_subject(slug, sj); moved.append(slug)
        if p := clean_group(g, c.subject):
            if p != c.get('group'): moved.append(slug)
            lr.set_group(slug, p, by='sort'); subjects.setdefault(c.subject, []).append(p)
    for s, gs in subjects.items(): lr.save_groups(s, gs, replace=True) if every else lr.save_groups(s)   # the groups of the saved sort stay
    return list(dict.fromkeys(moved))

def place_new(lr:Learning, ask, batch:int=None) -> list:
    "Place the new concepts of each subject that has groups: one request for each batch, and no second request. A concept that fits no group of the list, that has no subject, or that you undid stays. Return the slugs placed."
    todo = {c.slug for c in lr.unsorted() if c.subject and c.get('group_by') != 'undone' and lr.groups(c.subject)}
    out = []
    for s in sorted({lr.concept(x).subject for x in todo}):
        places = propose_groups(lr, ask, s, batch, only=todo, closed=True)   # no new group, and no second ask
        out += [slug for slug, g in places.items() if lr.place(slug, g)]   # it looks again, under the lock, just before it writes
    return out

These tests use five nakshatras, one concept that the learner moved by hand, and one concept in another subject. The fake model puts each concept that it sees in a group, and it also names a concept that it did not get and a concept that is not there.

In [ ]:
sl = temp_learning()
for k in range(5): sl.put(f'Nakshatra {k}', f'Star {k}.', 'jyotisha')
sl.put('Magha', 'The tenth nakshatra.', 'jyotisha'); sl.set_group('magha', 'mine')
sl.put('Sandhi', 'Sounds change where words meet.', 'sanskrit')
def sorter(p, sp):
    slugs = re.findall(r'^- ([a-z0-9-]+):', p, re.M)
    return dict(places=[dict(slug=s, group=['Nakshatras', f'Part {s[-1]}'] if s.startswith('nakshatra') else ['sound']) for s in slugs]
                + [dict(slug='magha', group=['taken']), dict(slug='nope', group=['x'])])
sfake = Fake({'Return JSON: {"places"': sorter})
places = propose_groups(sl, sfake, batch=2)

Five free nakshatras in batches of 2 give three requests, and Sanskrit gives one. The concept moved by hand is not in a request and not in the result. A slug that the model makes up does not count.

In [ ]:
test_eq(len(sfake.calls), 4)
test_eq(sorted(places), ['nakshatra-0', 'nakshatra-1', 'nakshatra-2', 'nakshatra-3', 'nakshatra-4', 'sandhi'])
test_eq((places['nakshatra-3'], places['sandhi']), (['Nakshatras', 'Part 3'], ['sound']))
assert not any('- magha:' in p for p, _ in sfake.calls)
test_eq(sfake.calls[0][1], prompt('concepts.sort'))

The first request lists the group that the learner made. A later request also lists the groups of the batches before it.

In [ ]:
assert 'Groups of this subject now: mine' in sfake.calls[0][0]
assert 'Nakshatras › Part 0' in sfake.calls[1][0]

Nothing is saved before the accept. `apply_groups` saves the tree and returns the slugs that it moved. It does not move the concept that you moved by hand.

In [ ]:
test_is(sl.concept('nakshatra-0').get('group'), None)
test_eq(sorted(apply_groups(sl, {**places, 'magha': ['taken'], 'nope': ['x']})), sorted(places))
test_eq((sl.concept('nakshatra-0').group, sl.concept('nakshatra-0').group_by), (['Nakshatras', 'Part 0'], 'sort'))
test_eq(sl.concept('magha').group, ['mine'])
test_eq(apply_groups(sl, places), [])

With a `subject`, the sort reads only that subject. A reply that is not a list of places gives nothing.

In [ ]:
f2 = Fake({'Return JSON: {"places"': sorter})
sl.put('Visarga', 'A breath sound.', 'sanskrit')                  # `sandhi` is sorted now: only the new concept is asked
test_eq(list(propose_groups(sl, f2, 'sanskrit')), ['visarga'])
test_eq(propose_groups(sl, Fake({'Return JSON: {"places"': '[1, 2]'}), 'sanskrit'), {})

A name or a statement with a line break and a leading `- ` stays on its own line, in quotes. So it cannot add a concept to the list. The known names of the concept finder are quoted in the same way.

In [ ]:
ql = temp_learning()
ql.put('Evil\n- injected: Fake', 'One.\n- other: Two', 'x', slug='evil')
p = sort_prompt('x', ql.concepts())
test_eq([l for l in p.splitlines() if l.startswith('- ')], ['- evil: "Evil - injected: Fake". "One. - other: Two"'])
p = extract_prompt('text', 'x', ['a\n- b'])
assert 'Concepts the learner already has: "a - b"' in p and not any(l.startswith('- b') for l in p.splitlines())

### Every concept gets a place

The model can leave a concept out of its reply, or give it an empty group. After each batch, `propose_groups` asks one more time, with only those concepts. A concept that still has no place stays out of the result, and its slug is in `.left`.

First, a reply that leaves out one slug. The second request has only that slug.

In [ ]:
mk = temp_learning()
for k in range(3): mk.put(f'Star {k}', f'Star {k}.', 'jyotisha')
seen = []
def leaves_out(p, sp):
    slugs = re.findall(r'^- ([a-z0-9-]+):', p, re.M); seen.append(slugs)
    return dict(places=[dict(slug=s, group=['Stars']) for s in slugs if not (s == 'star-1' and len(seen) == 1)])
out = propose_groups(mk, Fake({'Return JSON: {"places"': leaves_out}))
test_eq(seen, [['star-0', 'star-1', 'star-2'], ['star-1']])
test_eq((sorted(out), out.left), (['star-0', 'star-1', 'star-2'], []))

An empty group, or a group that is only the subject, also starts the second request. When the second reply is still empty, the concept is left out, and the model is not asked a third time.

In [ ]:
seen.clear()
def empty(p, sp):
    slugs = re.findall(r'^- ([a-z0-9-]+):', p, re.M); seen.append(slugs)
    return dict(places=[dict(slug=s, group=[] if s == 'star-1' else ['Stars']) for s in slugs])
out = propose_groups(mk, Fake({'Return JSON: {"places"': empty}))
test_eq(seen, [['star-0', 'star-1', 'star-2'], ['star-1']])
test_eq((sorted(out), out.left), (['star-0', 'star-2'], ['star-1']))
seen.clear()
def same(p, sp):
    slugs = re.findall(r'^- ([a-z0-9-]+):', p, re.M); seen.append(slugs)
    return dict(places=[dict(slug=s, group=['Jyotisha'] if s == 'star-1' and len(seen) == 1 else ['Stars']) for s in slugs])
out = propose_groups(mk, Fake({'Return JSON: {"places"': same}))
test_eq((len(seen), out.left, out['star-1']), (2, [], ['Stars']))

The prompt asks for a group for every concept, with a short `Other` group as the last resort.

In [ ]:
sp = prompt('concepts.sort')
assert 'Other' in sp and 'Use [] for a concept' not in sp

### A subject for a concept that has none

For concepts with no subject, the model also gives a subject: one that exists, in its own spelling, or a new short one. The sort asks about them as one batch. A concept that you moved by hand is not in it.

In [ ]:
ns = temp_learning()
ns.put('Rasi', 'A sign.', 'jyotisha'); ns.put('Attention', 'Weights.'); ns.put('Softmax', 'Scores to weights.'); ns.put('Pinned', 'Mine.'); ns.set_group('pinned', 'mine')
who = dict(attention='Jyotisha', softmax='  Transformers!')
def subj(p, sp):
    slugs = re.findall(r'^- ([a-z0-9-]+):', p, re.M)
    return dict(places=[dict(slug=s, group=['Stars'], **({'subject': who[s]} if s in who else {})) for s in slugs])
sfk = Fake({'Return JSON: {"places"': subj})
out = propose_groups(ns, sfk)
test_eq((out['attention'], out.subjects), (['Stars'], dict(attention='jyotisha', softmax='transformers')))
assert 'Subjects now: "jyotisha"' in sfk.calls[0][0] or 'Subjects now: "jyotisha"' in sfk.calls[1][0]
assert not any('- pinned:' in p for p, _ in sfk.calls) and 'pinned' not in out

The accept sets the subject first, as a move by the model, and then the group. A concept that you moved by hand keeps both, also when a result names it.

In [ ]:
test_eq(sorted(apply_groups(ns, out)), ['attention', 'rasi', 'softmax'])
test_eq((ns.concept('attention').subject, ns.concept('attention').group), ('jyotisha', ['Stars']))
test_eq((ns.concept('softmax').subject, ns.concept('softmax').subject_by), ('transformers', 'model'))
forced = Places({'pinned': ['q']}, subjects=dict(pinned='x'))
test_eq(apply_groups(ns, forced), [])
test_eq((ns.concept('pinned').subject, ns.concept('pinned').group), ('', ['mine']))

### Saved groups are locked

A sort only works on the concepts that have no group. A concept with a group stays where it is, and the groups that a sort saved stay in the list. The request has the group names of the subject, and the prompt tells the model to use one of them first.

In [ ]:
lk = temp_learning()
for k in range(3): lk.put(f'Star {k}', f'Star {k}.', 'jyotisha')
lk.put('Rasi', 'A sign.', 'jyotisha')
lk.set_group('star-0', ['Stars'], by='sort'); lk.set_group('rasi', ['Signs'], by='model'); lk.save_groups('jyotisha')
lseen = []
def lsorter(p, sp):
    sl_ = re.findall(r'^- ([a-z0-9-]+):', p, re.M); lseen.append((sl_, p))
    return dict(places=[dict(slug=s, group=['Stars']) for s in sl_])
lf = Fake({'Return JSON: {"places"': lsorter})
out = propose_groups(lk, lf)
test_eq([s for s, _ in lseen], [['star-1', 'star-2']])                 # only the two with no group
assert 'Groups of this subject now: Signs; Stars' in lseen[0][1]
assert 'Use a group from the list' in prompt('concepts.sort') and 'first' in prompt('concepts.sort')

The accept marks each concept as set by a saved sort and keeps the group list. A concept that got a group in the meantime does not move.

In [ ]:
lk.set_group('star-2', ['Mine too'], by='placed')
test_eq(sorted(apply_groups(lk, out)), ['star-1'])
test_eq((lk.concept('star-1').group, lk.concept('star-1').group_by, lk.concept('star-2').group), (['Stars'], 'sort', ['Mine too']))
test_eq(lk.saved_groups('jyotisha'), [['Mine too'], ['Signs'], ['Stars']])      # the saved list has the groups in use

### New concepts get a place by themselves

`place_new` places the new concepts of a subject that has groups. It makes one request for each batch, not one for each concept, and it asks no second time. It uses only a group that is in the list of the subject, spelled the same way. Any other answer, `Other` too, leaves the concept in `Not sorted yet`. A concept that you undid stays too.

In [ ]:
for k in range(3): lk.put(f'New {k}', f'New {k}.', 'jyotisha')
lk.put('Lone', 'No subject.'); lk.put('Gone', 'Undone.', 'jyotisha'); lk.set_group('gone', ['Stars'], by='placed'); lk.unplace('gone')
lseen.clear()
def lfit(p, sp):
    sl_ = re.findall(r'^- ([a-z0-9-]+):', p, re.M); lseen.append((sl_, p))
    return dict(places=[dict(slug=s, group={'new-2': ['Other'], 'new-1': ['stars']}.get(s, [' Stars '])) for s in sl_])
lf2 = Fake({'Return JSON: {"places"': lfit})
test_eq(place_new(lk, lf2), ['new-0'])                                   # `Other` is not in the list, and `stars` is not `Stars`
test_eq([s for s, _ in lseen], [['new-0', 'new-1', 'new-2']])                # one batch, and no second ask
test_eq((lk.concept('new-0').group, lk.concept('new-0').group_by), (['Stars'], 'placed'))
test_eq([lk.concept(s).get('group') for s in ('new-1', 'new-2', 'lone')], [None, None, None])
test_eq(lk.concept('gone').group, [])
assert 'Other' not in lk.groups('jyotisha') and ['Stars'] in lk.groups('jyotisha') and len(lk.groups('jyotisha')) == 3

A concept that `extract` adds to a subject with groups gets its group from the finder, marked as placed by the tutor, only when the group is in the list of the subject, spelled the same way. In a subject with no groups, the mark is `model`. A name such as `x` is not made.

In [ ]:
ex = temp_learning(); ex.put('Ashvini', 'The first.', 'jyotisha', group=['stars'], group_by='sort')
exf = Fake({'Return JSON: {"concepts"': dict(concepts=[dict(name='Bharani', statement='The second.', group=['stars']), dict(name='x', statement='A letter.'),
                                                       dict(name='Krittika', statement='The third.', group=['Stars']), dict(name='Rohini', statement='The fourth.', group=['Other'])])})
test_eq(sorted(extract(ex, exf, 'text', 'jyotisha')), ['bharani', 'krittika', 'rohini'])
test_eq([ex.concept(s).get('group') for s in ('krittika', 'rohini')], [None, None])   # `Stars` is not `stars`, and `Other` is not in the list
test_eq((ex.concept('bharani').group, ex.concept('bharani').group_by), (['stars'], 'placed'))
assert ex.concept('x') is None
extract(ex, Fake({'Return JSON: {"concepts"': dict(concepts=[dict(name='Atom', statement='Small.', group=['bits'])])}), 'text', 'physics')
test_eq(ex.concept('atom').group_by, 'model')

### Races

A placement is one step under a lock, and it looks at the concept again just before the write. If you undo, or sort the concept yourself, while the model works, the placement changes nothing. An undone concept is not placed again, also by a later `extract`. A re-import or a later `put` does not change a group that a saved sort, a placement or you set.

In [ ]:
rc = temp_learning()
for k in range(3): rc.put(f'Card {k}', f'Card {k}.', 'deck')
rc.set_group('card-0', ['Aces'], by='sort'); rc.save_groups('deck')
def meddle(p, sp):
    rc.update('card-1', group=[], group_by='undone')                                                       # you undo during the call
    rc.set_group('card-2', ['Aces'], by='sort')                                                              # you sort it during the call
    return dict(places=[dict(slug=s, group=['Aces']) for s in re.findall(r'^- ([a-z0-9-]+):', p, re.M)])
test_eq(place_new(rc, Fake({'Return JSON: {"places"': meddle})), [])
test_eq([(rc.concept(s).get('group'), rc.concept(s).get('group_by')) for s in ('card-1', 'card-2')], [([], 'undone'), (['Aces'], 'sort')])
test_eq(place_new(rc, Fake({'Return JSON: {"places"': meddle})), [])                         # `undone`: not placed again
test_eq(extract(rc, Fake({'Return JSON: {"concepts"': dict(concepts=[dict(name='Card 1', statement='Again.', group=['Aces'])])}), 'text', 'deck'), ['card-1'])
test_eq((rc.concept('card-1').get('group'), rc.concept('card-1').group_by), ([], 'undone'))

In [ ]:
for by, g in (('sort', ['Aces']), ('placed', ['Aces']), ('hand', ['Mine'])):
    rc.set_group('card-1', g, by=by)
    rc.put('Card 1', 'Again.', 'deck', slug='card-1', group=['Elsewhere'], group_by='model')      # a re-import, as from vedicreader
    test_eq((rc.concept('card-1').group, rc.concept('card-1').group_by), (g, by))

### Sort everything again

`propose_groups(every=True)` sorts every concept of a subject again. A concept that you moved by hand stays, except when its slug is in `free`: you let the tutor sort it too. The request lists only the groups that you made. `apply_groups(every=True)` saves the new tree and makes the group list again. The hand groups stay in the list.

In [ ]:
se = temp_learning()
for n in ('H1', 'H2'): se.put(n, f'{n}.', 'ml')
for n in ('S1', 'S2', 'S3'): se.put(n, f'{n}.', 'ml', group=['Old'], group_by='sort')
se.save_groups('ml'); se.new_group('ml', 'H')
for s in ('h1', 'h2'): se.set_group(s, ['H'])
sef = Fake({'Return JSON: {"places"': lambda p, sp: dict(places=[dict(slug=s, group=['New']) for s in re.findall(r'^- ([a-z0-9-]+):', p, re.M)])})
p = propose_groups(se, sef, 'ml', every=True)
test_eq(sorted(p), ['s1', 's2', 's3'])
assert 'Groups of this subject now: H\n' in sef.calls[-1][0] and 'Sort all these concepts again' in sef.calls[-1][0]

The accept moves the sorted concepts. The hand concepts stay, and the old group goes from the list.

In [ ]:
test_eq(sorted(apply_groups(se, p, every=True)), ['s1', 's2', 's3'])
test_eq((se.concept('h1').group, se.concept('s1').group, se.concept('s1').group_by), (['H'], ['New'], 'sort'))
test_eq(se.groups('ml'), [['H'], ['New']])

When you untick a group, its concepts are in `free`, and the tutor sorts them too.

In [ ]:
p = propose_groups(se, sef, 'ml', every=True, free=['h1', 'h2'])
test_eq(sorted(apply_groups(se, p, every=True, free=['h1', 'h2'])), ['h1', 'h2'])
test_eq((se.concept('h1').group, se.concept('h1').group_by), (['New'], 'sort'))

The accept is one step. When a write fails in the middle, the concepts, the group list and the hand marks are as they were before.

In [ ]:
at = temp_learning()
for n in ('A1', 'A2'): at.put(n, f'{n}.', 'ml', group=['Old'], group_by='sort')
at.save_groups('ml'); at.new_group('ml', 'Mine'); at.set_group('a2', ['Mine'])
calls = []
def breaks(slug, path, by='hand'):
    calls.append(slug)
    if len(calls) > 1: raise RuntimeError('the disk is full')
    return Learning.set_group(at, slug, path, by)
at.set_group = breaks
test_fail(lambda: apply_groups(at, {'a1': ['New'], 'a2': ['New']}, every=True, free=['a2'], unmark=[('ml', ['Mine'])]), contains='disk is full')
del at.set_group
test_eq([(at.concept(s).group, at.concept(s).group_by) for s in ('a1', 'a2')], [(['Old'], 'sort'), (['Mine'], 'hand')])
test_eq((at.hand_groups('ml'), at.saved_groups('ml')), ([['Mine']], [['Mine'], ['Old']]))

## From a link

You can give a link: an arXiv id or a PDF link, a docs page, or a GitHub repo or file. `file_link` files its pages in the vault, and the walk reads them in order.
abhyasa fetches a link only when you send the form, and only the link that you give. A crawl stays in the folder of the link and stops at the page cap.

The Links group of the settings sets the limits of a link: `links.cap` is the most pages that one link can give, `links.kept` is the most characters of one page that the vault keeps, and `links.read` is the most that the tutor reads.
`fetch_opts` gives the limits of each fetch from the settings: a time limit and a speed limit, so one page cannot be larger than 30 MB with the defaults.
It also gives `allow_private=False`, so that fossick refuses a private address on each fetch, also when `FOSSICK_ALLOW_PRIVATE` is set for Leela or for a test.
`link_folder` gives the folder of a link, `in_scope` tells if a link is in it, and `link_sel` is the CSS selector of the links that the crawl follows.

In [ ]:
#| export
import ipaddress, socket
from urllib.parse import unquote, urljoin, urlparse

def fetch_opts() -> dict:
    "The limits of one fetch, from the settings: the time limit, the speed cap in bytes a second and one retry. `allow_private=False` refuses a private address also when `$FOSSICK_ALLOW_PRIVATE` is set."
    return dict(timeout=val('links.timeout'), max_recv_speed=int(val('links.speed') * 1_000_000), retries=1, allow_private=False)

def link_folder(url:str) -> str:
    "The scheme, host and folder of `url`: all of it up to the last `/` of its path."
    u = urlparse(url)
    return f"{u.scheme}://{u.netloc}{u.path[:u.path.rfind('/') + 1] or '/'}"

def _plain(path:str) -> str:
    "`path` with its percent codes decoded, again until none is left, as a server can do."
    for _ in range(5):
        if (p := unquote(path)) == path: break
        path = p
    return path

def in_scope(url:str, start:str) -> bool:
    "Whether `url`, without its query and fragment, is in the folder of `start`. A path with `..`, also in percent codes, is never in it."
    u, path = urlparse(url), _plain(urlparse(url).path)
    return '..' not in re.split(r'[/\\]', path) and f"{u.scheme}://{u.netloc}{path}".startswith(_plain(link_folder(start)))

def link_sel(start:str) -> str:
    "The CSS selector of the links into the folder of `start`: from the root of the site, in full, or relative. None has `..` or `%2e`."
    f, up = link_folder(start).replace('"', '%22'), ':not([href*=".."]):not([href*="%2e" i])'
    return (f'a[href^="{urlparse(f).path}"]{up}, a[href^="{f}"]{up}, '
            f'a[href]:not([href*=":"]):not([href^="/"]):not([href^="#"]){up}')

`link_kind` names the reader for a link, as `fossick.what_is` does. It takes only an arXiv id, or an http or https link to a host on the internet.
The app must not fetch its own pages or the pages of the local network. `local_name` refuses a local name (such as `printer.local`), a name with no dot, and an address that is not on the internet, with no DNS look-up.
fossick has no rule for these names before DNS, so abhyasa keeps this one.
`public_url` uses `local_name`, then fossick's `check_url(url, allow_private=False)`: fossick looks up the host and raises `BlockedURL` for a private address, and `public_url` makes it a `LinkError`.
`file_link` uses `public_url` before it fetches, because a name on the internet can point to a local address.

The fetch checks again. With `allow_private=False` (from `fetch_opts`), fossick looks up the host, refuses a private address, and connects only to the addresses that it checked.
So a host that gives a different address each time cannot get past the check of the fetch.

In [ ]:
#| export
ARXIV_ID = re.compile(r'(?:\d{4}\.\d{4,5}|[a-z]+(?:-[a-z]+)?(?:\.[A-Z]{2})?/\d{7})(?:v\d+)?')   # 1706.03762, and the old form hep-th/9901001
LOCAL_ENDS = ('.localhost', '.local', '.internal', '.lan', '.home.arpa')
NOT_LOCAL = 'abhyasa does not read links to this computer or to a local network.'
NOT_FOUND = 'abhyasa could not find this site.'

class LinkError(ValueError):
    "A link that abhyasa does not read. Its words are for the learner."

def _ip(host):
    "`host` as an IP address, also in the short forms that the system takes, such as `127.1`; None for a name."
    try: return ipaddress.ip_address(host)
    except ValueError: pass
    try: return ipaddress.ip_address(socket.inet_aton(host))
    except OSError: return None

def local_name(host:str) -> bool:
    "Whether the name `host` is this computer or the local network, with no DNS look-up: a local name, a name with no dot, or an address that is not on the internet."
    h = host.lower().rstrip('.')
    if h == 'localhost' or h.endswith(LOCAL_ENDS) or ('.' not in h and ':' not in h): return True
    ip = _ip(h)
    return ip is not None and (not ip.is_global or ip.is_multicast)

def public_url(url:str) -> str:
    "`url` when abhyasa can fetch it: its host is not a `local_name`, and fossick's `check_url` finds only addresses on the internet. Else `LinkError`."
    from fossick import check_url, BlockedURL   # here, so that the import of this module stays light
    if local_name(urlparse(url).hostname or ''): raise LinkError(NOT_LOCAL)
    try: return check_url(url, allow_private=False)
    except BlockedURL as e: raise LinkError(NOT_FOUND if str(e).startswith('cannot resolve') else NOT_LOCAL) from e

def link_kind(url:str) -> str:
    "The reader for `url`, as `fossick.what_is` names it: only for an arXiv id, or an http or https link to a host on the internet. Else `LinkError`."
    url = url.strip()
    if ARXIV_ID.fullmatch(url): return 'arxiv'
    try: u = urlparse(url); host = u.hostname
    except ValueError: host = None
    if not host or u.scheme not in ('http', 'https'): raise LinkError('abhyasa reads only web links (http or https) and arXiv ids.')
    if local_name(host): raise LinkError(NOT_LOCAL)
    from fossick import what_is   # here, so that the import of this module stays light
    if (kind := what_is(url)) == 'youtube': raise LinkError('abhyasa cannot make a walk from a video link.')
    return kind

`reading_order` puts the pages of a docs site in the order of its menu. The start page comes first. Then come the pages in the order of their first link on the start page, and then the rest.

In [ ]:
#| export
def _bare(u): return urlparse(u)._replace(query='', fragment='').geturl().rstrip('/')

def reading_order(docs:list, start_text:str) -> L:
    "The start page, then the pages in the order of their first link in `start_text`, then the rest in their given order."
    if not docs: return L()
    at = {}
    for h in re.findall(r'\]\(\s*<?([^)\s>]+)', start_text or ''): at.setdefault(_bare(urljoin(docs[0].url, h)), len(at))
    return L([docs[0], *sorted(docs[1:], key=lambda d: at.get(_bare(d.url), len(at)))])

### Papers from arXiv HTML

The PDF of a paper loses its maths: "scale the dot products by $\frac{1}{\sqrt{d_k}}$" comes out as "by 1 dk". arXiv also gives most papers as an HTML page (made with LaTeXML), at `arxiv.org/html/<id>`. Each formula on that page has its TeX. So `file_link` reads that page first, and keeps each formula as `$…$` in the text. When the page is not there, it reads the PDF as before.

The page is text that abhyasa does not trust. abhyasa reads at most `research.PAGE_BYTES` of it, drops its scripts, styles and menus, and takes the TeX from the `alttext` (or the TeX annotation) of each MathML element, never as HTML. The vault keeps the text that the passages come from, so each passage stays word for word with its source.

`math_md` turns the page into markdown. Each formula becomes a token first, so that the markdown step cannot change its TeX. Then each token becomes `$TeX$`, or `$$TeX$$` for a block formula. A numbered equation (a LaTeXML table) becomes one line with its number. A `$` in the text becomes `\$`, so it does not start a formula.

In [ ]:
#| export
import hashlib, logging
log = logging.getLogger(__name__)

ARXIV_HTML, ARXIV_FIGS = 'https://arxiv.org/html/{aid}', 30   # the HTML page of a paper, and the most figures that abhyasa saves from it
_UNSAFE = ('script', 'style', 'noscript', 'template', 'iframe', 'object', 'embed', 'form', 'button', 'svg', 'nav', 'link', 'meta')
def _has(c): return f'contains(concat(" ", normalize-space(@class), " "), " {c} ")'

def _tex(m) -> str:
    "The TeX of the MathML element `m`: its `alttext`, else its TeX annotation, with no TeX comments, on one line, and with each `$` that is not escaped written `\\$`."
    t = m.get('alttext') or next((a.text or '' for a in m.iter('{*}annotation', 'annotation') if a.get('encoding') == 'application/x-tex'), '')
    return re.sub(r'(?<!\\)\$', r'\\$', ' '.join(re.sub(r'(?<!\\)%[^\n]*', '', t).split()))

def _swap(e, s:str):
    "Put the text `s` in the place of the element `e`."
    p, prev, s = e.getparent(), e.getprevious(), s + (e.tail or '')
    if prev is not None: prev.tail = (prev.tail or '') + s
    else: p.text = (p.text or '') + s
    p.remove(e)

def _ltx(s:str):
    "The `ltx_document` element of the arXiv page `s`, with no scripts, styles or menus and each formula as a token, and `{token: TeX}`. `(None, {})` for another page."
    import lxml.html
    root = first(lxml.html.fromstring(s).xpath(f'//*[{_has("ltx_document")}]'))
    if root is None: return None, {}
    for e in root.xpath('|'.join(f'.//{t}' for t in _UNSAFE) + f'|.//comment()|.//*[{_has("ltx_TOC")}]'): e.drop_tree()
    toks = {}
    def tok(t): k = f'ZXMATH{len(toks):05d}XZ'; toks[k] = t; return k
    for tb in root.xpath(f'.//table[{_has("ltx_equation")} or {_has("ltx_equationgroup")}]'):   # a numbered equation: a table with the formula and its number
        n = ' '.join(t.text_content().strip() for t in tb.xpath(f'.//*[{_has("ltx_tag_equation")}]'))
        p = lxml.html.Element('p'); p.tail = tb.tail
        p.text = ' '.join(tok(f'$${t}$$') for m in tb.xpath('.//*[local-name()="math"]') if (t := _tex(m))) + (f' {n}' if n else '')
        tb.getparent().replace(tb, p)
    for m in root.xpath('.//*[local-name()="math"]'):
        t = _tex(m)
        _swap(m, tok(f'$${t}$$' if m.get('display') == 'block' else f'${t}$') if t else '')
    return root, toks

def _untok(s:str, toks:dict) -> str:
    "`s` with each `$` written `\\$`, then each token as its TeX."
    return re.sub(r'ZXMATH\d{5}XZ', lambda m: toks.get(m[0], ''), s.replace('$', r'\$'))

def math_md(html:str) -> str:
    "The arXiv HTML page `html` as markdown, with each formula as its TeX: `$T$` inline and `$$T$$` for a block. `fossick.html2md` gives the rest of the text. '' for a page that is not a LaTeXML paper."
    import lxml.html
    from fossick import html2md
    root, toks = _ltx(html)
    return '' if root is None else _untok(html2md(lxml.html.tostring(root, encoding='unicode'), ignore_images=False, readability=False), toks).strip()

`read_arxiv_html` fetches the HTML page of paper `aid` one time, with plain HTTP and the limits of a link. It gives the title, the text and the links of the figures. It gives None on any error, for a status other than 200, for a page that is not a LaTeXML paper, and for a page with less than `SHORT` characters of text. A figure link must be in the folder of the paper on `arxiv.org/html`.

In [ ]:
#| export
def _figures(root, base:str, aid:str) -> list:
    "The links of the images under `root`, from the page `base`, that are in the HTML folder of paper `aid`: at most `ARXIV_FIGS`, each one time."
    at, out = re.compile(rf'https://arxiv\.org/html/{re.escape(aid)}(?:v\d+)?/'), []
    for src in root.xpath('.//img/@src'):
        u = urljoin(base, src.strip())
        if at.match(u) and in_scope(u, at.match(u)[0]) and urlparse(u).path.lower().endswith(IMG_SUFFIXES) and u not in out: out.append(u)
    return out[:ARXIV_FIGS]

def read_arxiv_html(aid:str, fetch=None) -> AttrDict|None:
    "The arXiv HTML page of paper `aid` as `title`, `text` (from `math_md`), `figures` (links) and `url`, from one plain fetch. None when there is no usable page."
    from abhyasa.research import PAGE_BYTES   # here: `research` imports this module
    if fetch is None: from fossick import fetch
    url = ARXIV_HTML.format(aid=aid)
    try:
        r = fetch(url, **PLAIN, **fetch_opts(), max_bytes=PAGE_BYTES)
        base = str(getattr(r, 'url', '') or url)
        if getattr(r, 'status', None) != 200 or urlparse(base).hostname != 'arxiv.org': return None
        b = (getattr(r, 'body', b'') or b'')[:PAGE_BYTES]
        s = b.decode('utf-8', 'replace') if isinstance(b, bytes) else str(b)
        if 'ltx_document' not in s: return None
        text, (root, toks) = math_md(s), _ltx(s)
        if root is None or len(text) < SHORT: return None
        h = first(root.xpath(f'.//h1[{_has("ltx_title_document")}]'))
        title = ' '.join(_untok(h.text_content(), toks).split()) if h is not None else ''
        return AttrDict(title=title[:200], text=text, figures=_figures(root, base, aid), url=base)
    except Exception: return None

`save_figures` saves each figure of the page in the assets folder of the paper, at the path that its link in the text gives, so `figure_path` finds it. It fetches from `arxiv.org` only, at most `research.PAGE_BYTES` for each figure. The address after the redirects must be `https` on `arxiv.org` or a subdomain of it. A larger figure is not saved, so no file is cut short. It keeps a file that is already there. `_file_html` files the text on the shelf of papers under `arxiv.org/abs/<id>`. When that source already has another text, as after a new version of the paper, the new text gets its own source key, so the passages of an old walk keep their source. `_paper_html` does both for `file_link`. A test gives its own `html`, so no figure is fetched.

In [ ]:
#| export
def save_figures(lr:Learning, aid:str, page, fetch=None) -> int:
    "Save each figure of the arXiv HTML `page` of paper `aid` in the assets folder of the paper. Return the count of new files."
    from abhyasa.research import PAGE_BYTES
    if fetch is None: from fossick import fetch
    base, n = link_folder(page.get('url') or ARXIV_HTML.format(aid=aid)), 0
    root = lr.vault.assets(aid.rsplit('/', 1)[-1]).resolve()   # the folder that `figure_path` reads for the source `arxiv.org/abs/<id>`
    for u in (page.get('figures') or ())[:ARXIV_FIGS]:
        if not (isinstance(u, str) and u.startswith(base) and urlparse(u).hostname == 'arxiv.org' and in_scope(u, base)): continue
        p = (root/_plain(u[len(base):].split('?')[0])).resolve()
        if not p.is_relative_to(root) or p.suffix.lower() not in IMG_SUFFIXES or p.is_file(): continue
        try: r = fetch(u, **PLAIN, **fetch_opts(), max_bytes=PAGE_BYTES)
        except Exception: continue
        f = urlparse(str(getattr(r, 'url', '') or u))   # the address after the redirects
        if f.scheme != 'https' or not (f.hostname == 'arxiv.org' or (f.hostname or '').endswith('.arxiv.org')): continue
        if getattr(r, 'status', None) != 200 or not isinstance(b := getattr(r, 'body', None), bytes) or not b or len(b) > PAGE_BYTES: continue   # never a file cut short
        p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(b); n += 1
    return n

def _file_html(lr:Learning, aid:str, page) -> AttrDict:
    "File the text of the arXiv HTML `page` of paper `aid` on the shelf of papers, never over another text. Return its `shelf`, `doc_id`, `title` and `url`."
    from vishalakshi.core import content_hash
    sh, text, src = lr.vault.shelf(lr.vault.route('arxiv').name), _clip(page.text), f'https://arxiv.org/abs/{aid}'
    if (d := first(sh.t.docs(where='source=?', where_args=[src]))) and json.loads(d['meta'] or '{}').get('content_hash') != content_hash(text):
        src += '?html=' + hashlib.sha1(text.encode()).hexdigest()[:10]   # a query keeps the link and the assets folder of the paper
    r = sh.add(text, page.get('title') or aid, source=src, kind='arxiv', refresh=False,
               meta=dict(url=page.get('url') or ARXIV_HTML.format(aid=aid), via='arxiv-html', aid=aid))
    d = sh.doc(r['doc_id'])
    return AttrDict(shelf=sh.name, doc_id=d['id'], title=d['title'], url=d['source'])

def _paper_html(lr:Learning, aid:str, page, title_of=None, figs:bool=True) -> L:
    "File the arXiv HTML `page` of paper `aid`, and with `figs` its figures. Return its one page, as `file_link` does. `title_of(id)` names a paper whose page has no title."
    d = _file_html(lr, aid, page)
    if figs: save_figures(lr, aid, page)
    if is_id_title(d.title, aid) and title_of and (t := title_of(aid)): d = AttrDict(d, title=t)
    return L([d])

`file_link` files a link with the reader that fits it, and returns the pages in reading order, at most the page cap.

| link | reader | what the walk uses |
|---|---|---|
| an arXiv id or link | `read_arxiv_html`, else `Vault.pdf` on the PDF of the paper | the paper, with its maths and its figures |
| a PDF link | `Vault.pdf` | the paper |
| a GitHub repo | `Vault.github` | its prose files, README first |
| a GitHub file | `Vault.gh_file` | the file |
| any other web page | `Vault.crawl`, or `Vault.url` when the cap is 1 | the start page and the pages in its folder |

`read(how, target, **kw)` does the fetch. The default calls the `Vault` method `how` on the shelf of the link, after it looks up the host.
A web page uses plain HTTP only (`auto=False`). A page that only a browser can read gives no page, because a browser runs the scripts of the page.
A docs site can redirect its start page to another host or folder, as `ai.pydantic.dev` and `docs.claude.com` do. For a crawl, `final(url)` fetches the start page one time and gives its address after the redirects.
The folder is taken from that address, and its host is checked like the first one: a host on this computer or the local network is refused. A redirect to a folder name such as `.../overview/` names the overview page, so the folder is the one above it. A locale folder such as `/docs/en/` or `/docs/pt-BR/` is the folder. A locale is a language code (ISO 639-1), so `/docs/go/` is not one. A redirect on the same host to a folder above the folder of the link, such as the bare host root, keeps the folder of the link.
Some pages lose almost all their text in the reader. `_whole` files such a page again with all its text. A link with almost no text in all its pages gives a message, not an empty walk.
A test gives a fake `read`, so the test does not use the network.

In [ ]:
#| export
READERS = dict(arxiv='arxiv', pdf='pdf', github='github', ghfile='gh_file')
PLAIN = dict(auto=False)   # a web page: plain HTTP only, so no browser and never your own Chrome

def _filed(r) -> list:
    "The documents in what a reader returned: one dict, a list of dicts, or a repo with its `docs`."
    rs = r['docs'] if isinstance(r, dict) and 'docs' in r else [r] if isinstance(r, dict) else r or []
    return [x for x in rs if isinstance(x, dict) and x.get('doc_id')]

def _clip(src):
    "Text, or `(page, text)` pairs, cut to `links.kept` characters."
    n = val('links.kept')
    return math_cut(src, n) if isinstance(src, str) else _cut(src, n)

LOCALE = re.compile(r'([a-z]{2})(?:[-_][A-Za-z]{2})?')   # a folder that names a language: en, en-us, pt-BR
LANGS = frozenset("""aa ab ae af ak am an ar as av ay az ba be bg bi bm bn bo br bs ca ce ch co cr cs cu cv cy da de dv dz ee el en eo es
et eu fa ff fi fj fo fr fy ga gd gl gn gu gv ha he hi ho hr ht hu hy hz ia id ie ig ii ik is it iu ja jv ka kg ki kj kk kl km kn ko kr
ks ku kv kw ky la lb lg li ln lo lt lu lv mg mh mi mk ml mn mr ms mt my na nb nd ne ng nl nn no nr nv ny oc oj om or os pa pi pl ps pt
qu rm rn ro ru rw sa sc sd se sg si sk sl sm sn so sq sr ss st su sv sw ta te tg th ti tk tl tn to tr ts tt tw ty ug uk ur uz ve vi vo
wa wo xh yi yo za zh zu""".split())   # the ISO 639-1 codes, less `io` (Ido), which a docs site uses for "I/O"

def _locale(name) -> bool:
    "Whether folder `name` names a language: a code in `LANGS`, with or without a region, such as `en`, `en-us` or `pt-BR`."
    return bool(m := LOCALE.fullmatch(name)) and m.group(1) in LANGS
SHORT = 200   # a page with less text than this was cut by readability: file it again with all its text
EMPTY = 40    # all the pages of a link with less text than this have nothing to read

def _whole(lr:Learning, sh, d:dict):
    "A web page `d` that readability cut to almost nothing: file it again with all its text, when its host is on the internet. A page that still has no text stays as it is."
    try: public_url(d['source'])
    except LinkError: return
    if sum(len(t) for _, t in doc_pages(lr, d['id'], sh.name).pages) >= SHORT: return
    from fossick import fetch, to_md   # here, so that the import of this module stays light
    md = to_md(fetch(d['source'], **PLAIN, **fetch_opts()), readability=False).strip()
    if len(md) >= SHORT: sh.add(_clip(md), d['title'], source=d['source'], kind='web', meta=dict(url=d['source'], via='whole'), force=True)

def _final(url:str) -> str:
    "The address of `url` after its redirects, from one plain fetch."
    from fossick import fetch
    return fetch(url, **PLAIN, **fetch_opts()).url

def _reader(lr:Learning, kind:str):
    "The default `read`: the `Vault` method `how` on a new copy of the shelf of `kind`, whose `add` keeps at most `links.kept` characters of a page."
    def read(how, target, **kw):
        sh = lr.vault.shelf(lr.vault.route(kind).name)   # a copy, so the shared vault does not change
        add = sh.add
        sh.add = lambda src, *a, **k: add(_clip(src), *a, **k)
        return getattr(sh, how)(target, **kw)
    return read

def file_link(lr:Learning, url:str, max_pages:int=None, read=None, final=None, title_of=None, html=None) -> L:
    "File the pages of `url` in the vault. Return `shelf`, `doc_id`, `title` and `url` of each page in the folder of `url` (after any redirect of the start page), in reading order, at most `max_pages` (default: the setting `links.pages`). `final(url)` gives the address after the redirects. A paper filed under its arXiv id gets its real title from `title_of(id)` (with the real readers, `arxiv_title`). An arXiv paper is read from `html(id)` first (with the real readers, `read_arxiv_html`), and from its PDF when that gives None."
    url, cap = url.strip(), max(1, min(int(max_pages or val('links.pages')), val('links.cap')))
    kind = link_kind(url)
    aid = (m := ARXIV_LINK.fullmatch(url) or ARXIV_ID.fullmatch(url)) and ARXIV_ID.search(m[0]).group()   # the id of an arXiv paper
    if kind == 'arxiv' and (m := ARXIV_ID.search(url)): kind, url = 'pdf', f'https://arxiv.org/pdf/{m.group()}'   # the arxiv reader gives the abstract only; the PDF gives the text and its figures
    how = READERS.get(kind) or ('url' if cap == 1 else 'crawl')
    found = read is not None
    if aid and (html or not found):
        if page := (html or read_arxiv_html)(aid): return _paper_html(lr, aid, page, title_of or (not found and arxiv_title), not found)
        log.warning('arxiv: no HTML page for %s, so abhyasa reads the PDF', aid)
    if read is None:
        public_url(url)
        read, final = _reader(lr, kind), final or _final
    start = scope = url
    if how == 'crawl':   # a docs site can redirect its start page to another host or folder: the folder is that of the page after the redirects
        start = (final or (lambda u: u))(url)
        if urlparse(start).scheme not in ('http', 'https') or local_name(urlparse(start).hostname or ''): raise LinkError(NOT_LOCAL)
        if not found: public_url(start)
        scope = start
        if start != url:
            path, last = urlparse(start).path, urlparse(start).path.rstrip('/').rsplit('/', 1)[-1]
            if path.endswith('/') and not _locale(last): scope = start.rstrip('/')   # `.../overview/` names the overview page, so its folder is the one above; a locale folder is the folder
            up, was = link_folder(scope), link_folder(url)
            if urlparse(start).netloc == urlparse(url).netloc and was != up and was.startswith(up): scope = url   # a redirect up the same host, also to its root, keeps the folder of the link
    f = fetch_opts()
    kw = dict(crawl=dict(max_pages=cap, follow_sel=link_sel(scope), **PLAIN, **f), url={**PLAIN, **f}, pdf=f, arxiv=f, github=dict(code=False)).get(how, {})
    shelves, docs = [lr.vault.route(kind), lr.vault], L()   # a reader can file on the shelf of the kind or on the main shelf
    for r in _filed(read(how, start, **kw)):
        sh, d = first(((s, s.doc(r['doc_id'])) for s in shelves), lambda x: x[1]) or (None, None)
        if not d or d['id'] in docs.attrgot('doc_id') or (how in ('url', 'crawl') and not in_scope(d['source'], scope)): continue
        if how in ('url', 'crawl') and not found: _whole(lr, sh, d)   # after the scope filter, and only for a host on the internet
        docs.append(AttrDict(shelf=sh.name, doc_id=d['id'], title=d['title'], url=d['source']))
    if aid and (t := (title_of or (not found and arxiv_title) or (lambda i: None))(aid)): docs = L(AttrDict(d, title=t) if is_id_title(d.title, aid) else d for d in docs)   # the PDF reader names a paper by the end of its link
    if how == 'github': docs = L(sorted(docs, key=lambda d: 'readme' not in Path(d.url).name.lower()))
    if not docs: raise LinkError('abhyasa could not read this link.')
    if sum(len(t) for d in docs for _, t in doc_pages(lr, d.doc_id, d.shelf).pages) < EMPTY:
        raise LinkError('abhyasa found almost no text on these pages. A page that needs a browser to show its text gives none.')
    return reading_order(docs, '\n\n'.join(t for _, t in doc_pages(lr, docs[0].doc_id, docs[0].shelf).pages))[:cap]


Each fetch refuses a private address, also when `FOSSICK_ALLOW_PRIVATE=1` is set. `_final` fetches `127.0.0.1` here, and fossick refuses it before it connects.

In [ ]:
import os, fossick

In [ ]:
was = os.environ.get('FOSSICK_ALLOW_PRIVATE'); os.environ['FOSSICK_ALLOW_PRIVATE'] = '1'
try:
    test_eq(fetch_opts()['allow_private'], False)
    test_fail(lambda: _final('http://127.0.0.1:9/'), exc=fossick.BlockedURL)
finally:
    if was is None: os.environ.pop('FOSSICK_ALLOW_PRIVATE', None)
    else: os.environ['FOSSICK_ALLOW_PRIVATE'] = was

The tests of links below use these imports.

In [ ]:
import fossick

In [ ]:
from fossick.core import Selector
from vishalakshi.core import Vault

`link_folder` gives the folder of a link. `in_scope` keeps a link in that folder. It refuses a link on a host that only starts with the same name, and a path with `..`, also when the `..` is in percent codes such as `%2e%2e`.

In [ ]:
START = 'https://docs.example.dev/en/docs/tool/overview'
test_eq(link_folder(START), 'https://docs.example.dev/en/docs/tool/')
test_eq(link_folder('https://ai.pydantic.dev'), 'https://ai.pydantic.dev/')
assert in_scope('https://docs.example.dev/en/docs/tool/setup#a', START) and not in_scope('https://docs.example.dev/en/api/x', START)
assert not in_scope('https://evil.example/en/docs/tool/x', START) and not in_scope('https://docs.example.dev.evil.example/en/docs/tool/', START)
assert not in_scope('https://docs.example.dev/en/docs/tool/../../admin', START)
for up in ('%2e%2e', '%2E%2E', '.%2e', '%2E.', '%252e%252e'):   # a `..` in percent codes, which many servers decode
    assert not in_scope(f'https://docs.example.dev/en/docs/tool/{up}/{up}/private', START), up
assert in_scope('https://docs.example.dev/en/docs/tool/my%20page', START) and in_scope('https://docs.example.dev/my%20docs/a', 'https://docs.example.dev/my docs/')

`link_sel` is the CSS selector of the links that the crawl follows: a link from the root of the site into the folder, a full link into the folder, and a relative link.
No part takes a link with `..` or `%2e`, so a link cannot go up out of the folder. The crawl also stays on the same host, and `in_scope` checks each page after the crawl.

In [ ]:
html = ''.join(f'<a href="{h}">x</a>' for h in ('agents/', '../models/', '/en/docs/tool/setup', 'https://docs.example.dev/en/docs/tool/hooks',
                                                  'https://evil.example/x', '#top', '/blog/news', '/en/docs/tool/../../admin', 'mailto:a@b.dev',
                                                  '%2e%2e/models/', '%2E%2E/models/', '.%2e/models/'))
test_eq([a.attrib['href'] for a in Selector(html).css(link_sel(START))],
        ['agents/', '/en/docs/tool/setup', 'https://docs.example.dev/en/docs/tool/hooks'])   # a relative link is followed; `..`, other hosts and other folders are not

`link_kind` takes only an arXiv id, or an http or https link to a host on the internet. It refuses a file, this computer and the local network.
It also refuses a video link. Each refusal is a `LinkError`, so the app can show its words to you.

In [ ]:
test_eq(link_kind('1706.03762'), 'arxiv')
test_eq(link_kind(START), 'web')
test_eq(link_kind('https://github.com/AnswerDotAI/fastcore'), 'github')
for bad in ('file:///etc/passwd', '/etc/passwd', 'ftp://example.com/a', 'http://localhost:5001/', 'http://127.0.0.1/', 'http://127.1/',
            'http://[::1]/', 'http://10.0.0.5/x', 'http://192.168.1.1/', 'http://169.254.169.254/latest', 'http://router/', 'http://nas.local/'):
    test_fail(lambda: link_kind(bad), exc=LinkError)
test_fail(lambda: link_kind('https://www.youtube.com/watch?v=x'), contains='video', exc=LinkError)

A learner can paste a link where the app asks for a vault document. `as_link` finds the link in a field: a web link, an arXiv id, `arxiv:<id>` or an arXiv address. A vault name gives None.
`is_paper` tells a paper from a site. A paper is one document that abhyasa reads in full, so a page count does not apply to it.

In [ ]:
#| export
ARXIV_LINK = re.compile(rf'(?:https?://)?(?:www\.|export\.)?arxiv\.org/(?:abs|pdf)/({ARXIV_ID.pattern})(?:\.pdf)?/?', re.I)

def as_link(s:str) -> str|None:
    "The link in the field `s`: a web link, an arXiv id, `arxiv:<id>` or an arXiv address. None for anything else, such as a vault name."
    s = re.sub(r'^arxiv:\s*', '', (s or '').strip(), flags=re.I)
    if ARXIV_ID.fullmatch(s) or re.fullmatch(r'https?://\S+', s, re.I): return s
    return f'https://arxiv.org/abs/{m[1]}' if (m := ARXIV_LINK.fullmatch(s)) else None

def is_id_title(t:str, aid:str='') -> bool:
    "Whether `t` is only an arXiv id, as the PDF reader names a paper by the end of its link. With `aid`, also its last part: `0601001` for `math/0601001`."
    t = re.sub(r'^arxiv:\s*', '', (t or '').strip(), flags=re.I)
    return bool(re.fullmatch(ARXIV_ID.pattern + r'(?:\.pdf)?', t) or aid and t.removesuffix('.pdf') == aid.rsplit('/', 1)[-1])

def arxiv_title(aid:str, strict:bool=False) -> str|None:
    "The title of arXiv paper `aid` from its metadata (fossick keeps it on disk), or None when there is none. `strict`: a failure raises, and the time limit is at most 10 s."
    from fossick import read_arxiv
    o = fetch_opts(); o['timeout'] = min(o['timeout'], 10) if strict else o['timeout']
    try: t = ' '.join(read_arxiv(aid, save_pdf=False, **o)['title'].split())
    except Exception:
        if strict: raise
        return None
    return (t if len(t) <= 200 else t[:200].rsplit(' ', 1)[0].rstrip() + '…') or None

def is_paper(url:str) -> bool:
    "Whether `url` is one paper that abhyasa reads in full: an arXiv id or address, or a link to a PDF file."
    u = (url or '').strip()
    if ARXIV_ID.fullmatch(u) or ARXIV_LINK.fullmatch(u): return True
    try: return urlparse(u).scheme in ('http', 'https') and urlparse(u).path.lower().endswith('.pdf')
    except ValueError: return False

A web link, an arXiv id and an arXiv address are links. A vault name or a title is not a link:

In [ ]:
test_eq(as_link(' https://arxiv.org/pdf/1706.03762 '), 'https://arxiv.org/pdf/1706.03762')
test_eq(as_link('1706.03762'), '1706.03762')
test_eq(as_link('arXiv: 1706.03762v2'), '1706.03762v2')
test_eq(as_link('arxiv.org/abs/1706.03762'), 'https://arxiv.org/abs/1706.03762')
for name in ('Rasi Characteristics', '3df4abc4deb9f8b0', '/tmp/attention.pdf', 'file:///etc/passwd', 'javascript:alert(1)', ''): test_is(as_link(name), None)

An arXiv link and a PDF link are papers. A docs site is not a paper:

In [ ]:
for u in ('1706.03762', 'https://arxiv.org/pdf/1706.03762', 'https://arxiv.org/abs/1706.03762v7', 'https://example.org/a/paper.PDF?x=1'): assert is_paper(u), u
for u in ('https://docs.example.dev/attention/', 'https://github.com/AnswerDotAI/fastcore', 'Rasi Characteristics', '/tmp/a.pdf', ''): assert not is_paper(u), u

`reading_order` puts the start page first, then the pages in the order of their first link on the start page. A page with no link stays at the end.

In [ ]:
pg = lambda u: AttrDict(url=f'https://d.dev/docs/{u}')
test_eq(reading_order([pg('a'), pg('x'), pg('c'), pg('b')], 'See [B](b#top), [C](/docs/c) and [B again](b).').attrgot('url'),
        [pg(u).url for u in 'abcx'])

A fake reader takes the place of the network. It files four pages: the start page, two pages in its folder, and one blog page outside it.

In [ ]:
SITE = {START: '# Overview\n\nSee [Setup](/en/docs/tool/setup) and [Hooks](/en/docs/tool/hooks).',
        'https://docs.example.dev/en/docs/tool/hooks': '# Hooks\n\nA hook runs a command at a set point.',
        'https://docs.example.dev/en/docs/tool/setup': '# Setup\n\nInstall the tool with one command.',
        'https://docs.example.dev/blog/news': '# News\n\nWe have a new logo.'}
calls = []
def fake_read(lr):
    "File every page of SITE in crawl order, and keep the call."
    def read(how, target, **kw):
        calls.append((how, target, kw))
        return [lr.vault.add(md, md.split('\n')[0][2:], source=u, kind='web', meta=dict(url=u)) for u, md in SITE.items()]
    return read

The crawl gets the folder and the page cap, and the walk does not use the blog page. A cap of 1 reads one page. An arXiv id uses the arXiv reader. A link with no pages gives a clear message.

In [ ]:
lr = temp_learning()
docs = file_link(lr, START, max_pages=500, read=fake_read(lr))
test_eq((calls[0][0], calls[0][2]['max_pages']), ('crawl', val('links.cap')))
assert 'a[href^="/en/docs/tool/"]' in calls[0][2]['follow_sel']
test_eq([d.title for d in docs], ['Overview', 'Setup', 'Hooks'])        # the start page, then its links in order; no blog page
test_eq(docs[0].shelf, 'store')
calls.clear(); test_eq(len(file_link(lr, START, max_pages=1, read=fake_read(lr))), 1); test_eq(calls[0][0], 'url')
calls.clear(); file_link(lr, '1706.03762', read=fake_read(lr)); test_eq(calls[0][:2], ('pdf', 'https://arxiv.org/pdf/1706.03762'))   # the PDF, not the abstract
test_fail(lambda: file_link(temp_learning(), START, read=lambda how, target, **kw: []), contains='could not read')

A paper that the PDF reader filed under its arXiv id gets the real title of the paper. A paper that has a real title keeps it.


In [ ]:
TEXT = '# Attention\n\nSee the paper. ' * 30
def paper(url, title, **kw):
    "The title of the one page that `file_link` gives for a paper that a reader filed under `title`."
    l = temp_learning()
    return file_link(l, url, read=lambda how, target, **k: [l.vault.add(TEXT, title, source=target, kind='pdf')], **kw)[0].title
test_eq(paper('1706.03762', '1706.03762', title_of=lambda i: 'Attention Is All You Need'), 'Attention Is All You Need')
test_eq(paper('https://arxiv.org/abs/1706.03762v7', '1706.03762v7.pdf', title_of=lambda i: i + ' paper'), '1706.03762v7 paper')   # the id with its version
test_eq(paper('1706.03762', 'Attention', title_of=lambda i: 'Other'), 'Attention')
test_eq(paper('1706.03762', '1706.03762', title_of=lambda i: None), '1706.03762')   # no title to give: the id stays
test_eq(paper('hep-th/9901001v2', '9901001v2.pdf', title_of=lambda i: 'Old ' + i), 'Old hep-th/9901001v2')   # the old form of an id
test_eq(paper('https://arxiv.org/abs/math.GT/0601001', '0601001', title_of=lambda i: 'Old ' + i), 'Old math.GT/0601001')
test_eq([ARXIV_LINK.fullmatch(u)[1] for u in ('https://arxiv.org/pdf/math/0601001.pdf', 'arxiv.org/abs/hep-th/9901001v2')], ['math/0601001', 'hep-th/9901001v2'])
test_eq([as_link(s) for s in ('math/0601001', 'arXiv:hep-th/9901001v2')], ['math/0601001', 'hep-th/9901001v2'])
test_eq(is_paper('math/0601001'), True); test_eq(is_paper('docs/2024'), False)
test_eq(is_id_title('0601001', 'math/0601001'), True); test_eq(is_id_title('0601001'), False)
from unittest.mock import patch as mock_patch
with mock_patch('fossick.read_arxiv', return_value=dict(title='word ' * 80)): t = arxiv_title('1706.03762')
assert len(t) <= 201 and t.endswith('word…')   # cut at a word
with mock_patch('fossick.read_arxiv', return_value=dict(title='  A  short\ntitle ')): test_eq(arxiv_title('1706.03762'), 'A short title')
test_eq([is_id_title(t) for t in ('1706.03762', 'arXiv:1706.03762v5', '1706.03762.pdf', 'Attention', '', None)], [True, True, True, False, False, False])

Before it fetches, `file_link` looks up the host. A host name that points to this computer or to the local network is refused. Here a fake lookup gives the address.

In [ ]:
def fake_dns(*addrs):
    "A `socket.getaddrinfo` that gives each name the addresses `addrs`, so a test does no look-up."
    return lambda host, port, *a, **k: [(socket.AF_INET, socket.SOCK_STREAM, 6, '', (x, port or 0)) for x in addrs]
def no_dns(*a, **k): raise socket.gaierror('no such name')

was, socket.getaddrinfo = socket.getaddrinfo, fake_dns('127.0.0.1')
os.environ['FOSSICK_ALLOW_PRIVATE'] = '1'   # the check refuses a private address also with this variable set
try:
    test_fail(lambda: file_link(temp_learning(), 'https://evil.example/docs/'), contains='local network')
    socket.getaddrinfo = fake_dns('93.184.216.34'); test_eq(public_url('https://ok.example/docs/'), 'https://ok.example/docs/')
    socket.getaddrinfo = no_dns; test_fail(lambda: public_url('https://gone.example/'), contains='could not find')
finally:
    socket.getaddrinfo = was; os.environ.pop('FOSSICK_ALLOW_PRIVATE')

A docs site can redirect its start page to another host, or to another host and folder. `file_link` takes the folder from the page after the redirects. It refuses a redirect to a local host.


In [ ]:
REDIRECTS = {'https://old.example.dev/': 'https://new.example.dev/docs/ai/overview/',
             'https://docs.example.dev/en/docs/tool/overview': 'https://code.example.dev/docs/en/overview'}
def redirect_read(lr, to):
    "A fake `read` for a site that sends its start page to `to`; the pages of the site are in the folder of `to`."
    base = to.rsplit('/', 1)[0] + '/'
    site = {to: '# Overview\n\nSee [Setup](setup).', base + 'setup': '# Setup\n\nInstall it.', 'https://elsewhere.example/x': '# Other\n\nNot in the folder.'}
    def read(how, target, **kw):
        calls.append((how, target, kw))
        return [lr.vault.add(md, md.split('\n')[0][2:], source=u, kind='web') for u, md in site.items()]
    return read
for start, to in REDIRECTS.items():
    calls.clear(); lr = temp_learning()
    test_eq([d.title for d in file_link(lr, start, 8, read=redirect_read(lr, to), final=lambda u: to)], ['Overview', 'Setup'])   # the pages of the new folder
    test_eq(calls[-1][1], to)                                                                              # the crawl starts at the page after the redirect
    assert urlparse(to.rstrip('/')).path.rsplit('/', 1)[0] + '/' in calls[-1][2]['follow_sel']             # `.../overview/` gives the folder above it
for bad in ('http://127.0.0.1:8080/docs/x', 'http://intranet/docs/x'):
    lr = temp_learning()
    test_fail(lambda: file_link(lr, 'https://old.example.dev/', read=redirect_read(lr, bad), final=lambda u: bad), contains='local network')
def scoped(start, to, extra):
    "The titles that `file_link` keeps when `start` is sent to `to`, and the site has the pages `extra` (address -> markdown) as well."
    lr = temp_learning(); base = to.rsplit('/', 1)[0] + '/'
    def read(how, target, **kw):
        site = {to: '# Overview\n\nSee more.', **extra}
        return [lr.vault.add(md + ' Text of the page.' * 4, md.split('\n')[0][2:], source=u, kind='web') for u, md in site.items()]
    return [d.title for d in file_link(lr, start, 8, read=read, final=lambda u: to)]
# a locale folder is the folder: /docs/en/ keeps its pages and drops /docs/fr/ and /docs/
test_eq(scoped('https://old.example.dev/', 'https://new.example.dev/docs/en/', {'https://new.example.dev/docs/en/setup': '# Setup\n\nInstall it.', 'https://new.example.dev/docs/fr/setup': '# Setup fr\n\nInstaller.', 'https://new.example.dev/docs/blog': '# Blog\n\nNews.'}), ['Overview', 'Setup'])
test_eq(scoped('https://old.example.dev/', 'https://new.example.dev/docs/en-US/', {'https://new.example.dev/docs/en-US/setup': '# Setup\n\nInstall it.', 'https://new.example.dev/docs/blog': '# Blog\n\nNews.'}), ['Overview', 'Setup'])
# a redirect to the bare root of the same host keeps the folder of the link
test_eq(scoped('https://docs.example.dev/en/docs/tool/overview', 'https://docs.example.dev/', {'https://docs.example.dev/en/docs/tool/setup': '# Setup\n\nInstall it.', 'https://docs.example.dev/blog': '# Blog\n\nNews.'}), ['Setup'])


A two-letter folder is a locale only when it is a language code (ISO 639-1), such as `en`, `fr` or `pt-BR`. A folder such as `go`, `io` or `ui` is a folder name, so the folder of the link is the one above it.

In [ ]:
for name in ('go', 'io', 'ui', 'js'): test_eq(scoped('https://old.example.dev/', f'https://new.example.dev/docs/{name}/', {'https://new.example.dev/docs/setup': '# Setup\n\nInstall it.'}), ['Overview', 'Setup'])
assert all(map(_locale, ('en', 'fr', 'en-us', 'pt-BR', 'zh_CN'))) and not any(map(_locale, ('go', 'io', 'ui', 'xx-yy', 'eng', 'EN')))

A redirect to a folder above the folder of the link, on the same host, keeps the folder of the link. Thus `/docs/x/` sent to `/docs/` reads only the pages in `/docs/x/`, and not the whole site.

In [ ]:
extra = {'https://docs.example.dev/docs/x/setup': '# Setup\n\nInstall it.', 'https://docs.example.dev/docs/blog': '# Blog\n\nNews.', 'https://docs.example.dev/about': '# About\n\nUs.'}
for to in ('https://docs.example.dev/docs/', 'https://docs.example.dev/docs/index.html', 'https://docs.example.dev/'):
    test_eq(scoped('https://docs.example.dev/docs/x/', to, extra), ['Setup'])
test_eq(scoped('https://docs.example.dev/docs/x/', 'https://other.example.dev/docs/', {'https://other.example.dev/docs/a': '# A\n\nText.'}), ['Overview', 'A'])   # another host: its folder

A page that readability cut to almost nothing is filed again with all its text. A link with no text at all gives a message.


In [ ]:
WHOLE = '# Docs\n\n' + 'The tool has many features. ' * 20
class Page: url = START
real = fossick.fetch, fossick.to_md
fossick.fetch, fossick.to_md = lambda u, **kw: Page(), lambda p, readability=True, **kw: WHOLE if not readability else ''
was_crawl, was_dns, was_final = Vault.crawl, socket.getaddrinfo, _final
Vault.crawl = lambda self, u, **kw: L([self.add('# Docs', 'Docs', source=START, kind='web')])   # readability kept 7 characters
socket.getaddrinfo, _final = fake_dns('93.184.216.34'), lambda u: u
try:
    lr = temp_learning(); d = file_link(lr, START, 8)[0]
    assert len(doc_pages(lr, d.doc_id).pages[0][1]) > SHORT                                             # filed again with all its text
    Vault.crawl = lambda self, u, **kw: L([self.add('# X', 'X', source=START + 'x', kind='web')])
    fossick.to_md = lambda p, readability=True, **kw: ''
    test_fail(lambda: file_link(temp_learning(), START, 8), contains='almost no text')                  # a clear message, not an empty walk
    fetched = []
    fossick.fetch = lambda u, **kw: fetched.append(u) or Page()
    Vault.crawl = lambda self, u, **kw: L([self.add('# In', 'In', source=START + 'a', kind='web'), self.add('# Out', 'Out', source='https://docs.example.dev/blog/b', kind='web')])
    fossick.to_md = lambda p, readability=True, **kw: WHOLE if not readability else ''
    file_link(temp_learning(), START, 8); test_eq(fetched, [START + 'a'])                                # a page outside the folder is not fetched again
    hosts = iter(['93.184.216.34', '93.184.216.34', '127.0.0.1']); fetched.clear()   # the look-ups: the link, the start page, then the page
    socket.getaddrinfo = lambda *a, **k: fake_dns(next(hosts))(*a, **k)                                  # the host now points to this computer
    test_fail(lambda: file_link(temp_learning(), START, 8), contains='almost no text'); test_eq(fetched, [])
finally:
    fossick.fetch, fossick.to_md = real; Vault.crawl, socket.getaddrinfo, _final = was_crawl, was_dns, was_final

`_cut` keeps the first characters of the pages, for the cap on one linked page.

In [ ]:
test_eq(_cut([(1, 'abc'), (2, 'defg'), (3, 'h')], 5), [(1, 'abc'), (2, 'de')])
test_eq(_cut([(1, 'abc')], None), [(1, 'abc')])

The default reader uses plain HTTP only. It never starts a browser and never uses your own Chrome. Each fetch has a time limit and a speed limit, so one page cannot be very large.
The vault keeps at most `links.kept` characters of a page. Here fake `Vault` readers keep their keywords and file one page of 500 kB.

In [ ]:
READS = ('url', 'crawl', 'pdf', 'arxiv', 'gh_file', 'github')
got, real = {}, {k: getattr(Vault, k) for k in READS}
BIG = '# Big\n\n' + 'word ' * 100_000
def fake_vault(how):
    "A `Vault` reader that keeps its keywords and files one big page."
    def f(self, target, **kw):
        got[how] = kw
        r = self.add(BIG, 'Big', source=target if how in ('url', 'crawl') else f'{target}#{how}', kind='web')
        return L([r]) if how == 'crawl' else AttrDict(docs=L([r])) if how == 'github' else r
    return f
for k in READS: setattr(Vault, k, fake_vault(k))
was, socket.getaddrinfo, was_final, _final = socket.getaddrinfo, fake_dns('93.184.216.34'), _final, lambda u: u
was_html, read_arxiv_html = read_arxiv_html, lambda aid, fetch=None: None   # no HTML page, so the paper is read as a PDF
try:
    lr = temp_learning()
    for u, n in [(START, 1), (START, 8), ('https://x.example/paper.pdf', 8), ('1706.03762', 8),
                 ('https://github.com/a/b/blob/main/README.md', 8), ('https://github.com/a/b', 8)]: file_link(lr, u, n)
    test_eq(sorted(got), sorted(set(READS) - {'arxiv'}))   # an arXiv id is read as a PDF
    test_eq((got['url']['auto'], got['crawl']['auto']), (False, False))           # plain HTTP: no browser, no logged-in Chrome
    assert not any(kw.get(k) for kw in got.values() for k in ('heavy', 'stealthy', 'session'))
    assert all(got[k]['max_recv_speed'] == fetch_opts()['max_recv_speed'] and got[k]['timeout'] == fetch_opts()['timeout'] for k in ('url', 'crawl', 'pdf'))
    kept = sum(len(t) for _, t in doc_pages(lr, file_link(lr, START, 1)[0].doc_id).pages)
    assert kept <= val('links.kept') * 1.05, kept                                          # the big page is cut before the vault keeps it
finally:
    for k, f in real.items(): setattr(Vault, k, f)
    socket.getaddrinfo, _final, read_arxiv_html = was, was_final, was_html

### Tests of the arXiv HTML reader

A small saved page in the LaTeXML form takes the place of the network. It has a title, two paragraphs, an inline formula, a numbered equation, a script and one figure.

In [ ]:
import abhyasa

In [ ]:
SAMPLE = (Path(abhyasa.__file__).parents[1]/'tests/data/arxiv_html_sample.html').read_text()

`math_md` keeps the TeX of each formula. The numbered equation is one `$$…$$` line with its number. The script, the menu and the PDF form "1 dk" are not in the text, and the `$` of a price is not a formula.

In [ ]:
md = math_md(SAMPLE)
assert r'scale the dot products by $\frac{1}{\sqrt{d_k}}$ before the softmax' in md and '1 dk' not in md
assert r'$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}(\frac{QK^{T}}{\sqrt{d_{k}}})V$$ (1)' in md
assert 'alert' not in md and 'not the paper' not in md and '3 Model Architecture' not in md   # no script, no menu
assert r'A price of \$5 is not maths.' in md and '![Refer to caption](x1.png)' in md
test_eq(math_md('<html><body><p>Not a paper.</p></body></html>'), '')
test_eq(math_md('<article class="ltx_document"><p>Pay <math alttext="\\$5 + x$" display="inline"></math> now.</p></article>'), r'Pay $\$5 + x\$$ now.')   # a TeX `\$` stays one escape

`read_arxiv_html` makes one plain fetch, with the limits of a link and no private host. It gives the title, the text, and the figure in the folder of the paper. An image outside that folder is not a figure.

In [ ]:
from abhyasa.research import PAGE_BYTES

In [ ]:
class Resp:
    def __init__(self, body, status=200, url='https://arxiv.org/html/1706.03762v7/'): self.body, self.status, self.url = body, status, url
seen = []
def page_fetch(body=SAMPLE, **kw):
    "A fake fetch that gives `body` and records its call."
    def f(u, **o): seen.append((u, o)); return Resp(body.encode() if isinstance(body, str) else body, **kw)
    return f
p = read_arxiv_html('1706.03762', fetch=page_fetch(SAMPLE.replace('</figure>', '<img src="/static/logo.png"><img src="../1706.03763/x.png"></figure>')))
test_eq((p.title, p.figures, p.url), ('Attention Is All You Need', ['https://arxiv.org/html/1706.03762v7/x1.png'], 'https://arxiv.org/html/1706.03762v7/'))
assert r'$\frac{1}{\sqrt{d_k}}$' in p.text
test_eq(seen[0][0], 'https://arxiv.org/html/1706.03762')
test_eq((seen[0][1]['auto'], seen[0][1]['allow_private'], seen[0][1]['max_bytes']), (False, False, PAGE_BYTES))

It gives None for a status other than 200, for a page that is not a LaTeXML paper, for a page with almost no text, for a redirect to another host, and for an error.

In [ ]:
def boom(u, **o): raise OSError('no network')
test_is(read_arxiv_html('1706.03762', fetch=page_fetch(status=404)), None)
test_is(read_arxiv_html('1706.03762', fetch=page_fetch('<html><body>' + 'Text. ' * 100 + '</body></html>')), None)
test_is(read_arxiv_html('1706.03762', fetch=page_fetch('<article class="ltx_document"><p>Short.</p></article>')), None)
test_is(read_arxiv_html('1706.03762', fetch=page_fetch(url='https://evil.example/html/1706.03762')), None)
test_is(read_arxiv_html('1706.03762', fetch=boom), None)

`file_link` files the HTML text of a paper under `arxiv.org/abs/<id>` on the shelf of papers, and does not read the PDF. The text keeps the maths.

In [ ]:
def no_pdf(how, target, **kw): raise AssertionError('the PDF was read')
html_of = lambda aid: AttrDict(title='Attention Is All You Need', text=math_md(SAMPLE), figures=[], url=ARXIV_HTML.format(aid=aid))
lr = temp_learning(); d = file_link(lr, '1706.03762', html=html_of, read=no_pdf)[0]
test_eq((d.url, d.title, d.shelf), ('https://arxiv.org/abs/1706.03762', 'Attention Is All You Need', 'papers'))
text = '\n\n'.join(t for _, t in doc_pages(lr, d.doc_id, d.shelf).pages)
assert r'$\frac{1}{\sqrt{d_k}}$' in text
test_eq(lr.vault.shelf('papers').doc(d.doc_id)['kind'], 'arxiv')

With no HTML page, `file_link` reads the PDF as before. A PDF copy that the vault has stays as it is, and a new HTML copy goes beside it. A changed HTML text gets its own source key, so the old text stays too.

In [ ]:
pdf = file_link(lr, 'https://arxiv.org/pdf/1706.03762', html=lambda aid: None, read=lambda how, target, **k: [lr.vault.add(TEXT, 'Attention', source=target, kind='pdf')])[0]
test_eq(pdf.url, 'https://arxiv.org/pdf/1706.03762')
d2 = file_link(lr, '1706.03762', html=html_of, read=no_pdf)[0]; test_eq(d2.doc_id, d.doc_id)        # the same text: the same document
d3 = file_link(lr, '1706.03762', html=lambda aid: AttrDict(html_of(aid), text=math_md(SAMPLE) + '\n\nA new version.'), read=no_pdf)[0]
assert d3.doc_id != d.doc_id and d3.url.startswith('https://arxiv.org/abs/1706.03762?html=')
assert r'\sqrt{d_k}' in doc_pages(lr, d.doc_id, 'papers').pages[0][1] and doc_pages(lr, pdf.doc_id).pages

`save_figures` saves a figure at the path of its link in the text, so `figure_path` finds it for the paper. It fetches only from the folder of the page, and keeps a file that is already there.

In [ ]:
got = []
def img_fetch(u, **o): got.append(u); return Resp(b'\x89PNG\r\n\x1a\n', url=u)
pg = AttrDict(url='https://arxiv.org/html/1706.03762v7/', figures=['https://arxiv.org/html/1706.03762v7/x1.png', 'https://evil.example/x2.png', 'https://arxiv.org/html/1706.03762v7/x3.svg'])
test_eq(save_figures(lr, '1706.03762', pg, fetch=img_fetch), 1); test_eq(got, ['https://arxiv.org/html/1706.03762v7/x1.png'])
test_eq(figure_path(lr, d.url, 'x1.png'), '1706.03762/x1.png')
test_eq(save_figures(lr, '1706.03762', pg, fetch=img_fetch), 0)                                  # the file is there: no fetch
test_eq(save_figures(lr, '1706.03762', AttrDict(pg, figures=['https://arxiv.org/html/1706.03762v7/y.png']), fetch=lambda u, **o: Resp(b'', status=404)), 0)
one = lambda name: AttrDict(pg, figures=[f'https://arxiv.org/html/1706.03762v7/{name}.png'])
for bad in ('https://evil.example/y.png', 'http://arxiv.org/y.png', 'https://arxiv.org.evil.example/y.png'):   # a redirect away from https arxiv.org
    test_eq(save_figures(lr, '1706.03762', one('r'), fetch=lambda u, **o: Resp(b'\x89PNG', url=bad)), 0)
test_eq(save_figures(lr, '1706.03762', one('s'), fetch=lambda u, **o: Resp(b'\x89PNG', url='https://static.arxiv.org/s.png')), 1)   # a subdomain of arxiv.org
test_eq(save_figures(lr, '1706.03762', one('big'), fetch=lambda u, **o: Resp(b'x' * (PAGE_BYTES + 1), url=u)), 0)            # too big: not saved, not cut
assert not (lr.vault.assets('1706.03762')/'big.png').exists() and not (lr.vault.assets('1706.03762')/'r.png').exists()

## From a folder or a repo

A goal can name a folder on this Mac, a repo on this Mac or a public GitHub repo. abhyasa files its text in the vault, on
the `notebooks` shelf, so a walk can quote it. It reads only text: it never runs the code of a folder or a repo.

- `file_folder` files a folder in your home folder. `.ipynb` goes through `index_notebooks`, and `.md`, `.txt` and `.py`
  go through the vault, as text.
- `file_repo` clones a public GitHub repo with fossick's `gh_clone` (HTTPS only), then files the clone. abhyasa has no clone code of its own.
- Each document keeps `meta.root` (the folder name, or `owner/name`) and `meta.path`. A GitHub file also keeps
  `meta.url`, its page on GitHub, so its source line links there and not to the cache.
- `unfile` forgets what one filing added. This is the Undo after Start.

In [ ]:
#| export
from urllib.parse import quote
from abhyasa.ground import SKIP_DIRS, index_notebooks

### The files that abhyasa reads

A folder is text that abhyasa does not trust. `tree_files` keeps only `.ipynb`, `.md`, `.txt` and `.py` files, each at most
`TREE_MB` MB, and at most `TREE_FILES` files, notebooks first. It skips `.git`, `.venv`, `node_modules`, data folders and
dot folders. It skips a link, and it keeps no file whose real path is outside the folder. It looks at no more than
`TREE_SEEN` names, so a very big folder does not stop the app.

In [ ]:
#| export
TREE_TYPES = ('.ipynb', '.md', '.txt', '.py')   # in this order: the caps keep the notebooks first
TREE_SKIP = SKIP_DIRS | {'data', 'datasets'}
TREE_FILES, TREE_MB, TREE_TOTAL_MB, TREE_SEEN = 400, 20, 200, 20_000   # the most files, MB of one file, MB of one filing, names looked at

def tree_ok(p, root, most:int=None) -> bool:
    "Whether abhyasa may read the file `p` under the folder `root`: not a link, not a hard link, at most `most` bytes (default `TREE_MB` MB), and inside `root` after each link is resolved."
    p, most = Path(p), TREE_MB * 2**20 if most is None else most
    try:
        if p.is_symlink(): return False
        st = p.lstat()   # a hard link can name a file outside the folder, such as a key: it is never read
        return st.st_nlink == 1 and st.st_size <= most and p.resolve().is_relative_to(Path(root).resolve())
    except OSError: return False

def tree_files(root) -> L:
    """The files under the folder `root` that abhyasa files, in path order. No link, no hard link, nothing outside `root`,
    no `.py` when the folder has notebooks, at most `TREE_FILES` files and `TREE_TOTAL_MB` MB. `cut` says why it stopped early, or is ''."""
    root, got, seen, nbs = Path(root).resolve(), [], 0, False
    for dp, dns, fns in os.walk(root):   # `os.walk` does not go into a link to a folder
        dns[:] = sorted(d for d in dns if d not in TREE_SKIP and not d.startswith('.'))
        for f in sorted(fns):
            seen += 1
            p = Path(dp)/f
            if f.startswith('.') or p.suffix.lower() not in TREE_TYPES or p.is_symlink(): continue
            nbs = nbs or p.suffix.lower() == '.ipynb'
            if tree_ok(p, root):
                try: got.append((p, p.lstat().st_size))
                except OSError: continue
        if seen >= TREE_SEEN: break
    if nbs: got = [(p, n) for p, n in got if p.suffix.lower() != '.py']   # the code of a course is in its notebooks
    got.sort(key=lambda x: (TREE_TYPES.index(x[0].suffix.lower()), str(x[0])))
    keep, total, cut = [], 0, ''
    for p, n in got:
        if len(keep) == TREE_FILES: cut = f'abhyasa stopped at {TREE_FILES} files.'; break
        if total + n > TREE_TOTAL_MB * 2**20: cut = f'abhyasa stopped at {TREE_TOTAL_MB} MB.'; break
        keep.append(p); total += n
    out = L(sorted(keep))
    out.cut = cut
    return out

A small folder: two notebooks, a note, a data folder, a `.git` folder, a file of another type, and a link to a file outside the folder.

In [ ]:
import tempfile
from fastcore.nbio import new_nb, mk_cell, write_nb

In [ ]:
def two_notebooks(d):
    "Write `01_a.ipynb` and `02_b.ipynb` in the folder `d`."
    write_nb(new_nb([mk_cell('# Noise\n\nA diffusion model adds noise, then learns to remove it.', 'markdown'), mk_cell('x = 1')]), Path(d)/'01_a.ipynb')
    write_nb(new_nb([mk_cell('# Steps\n\nThe sampler removes the noise in small steps.', 'markdown')]), Path(d)/'02_b.ipynb')

top = Path(tempfile.mkdtemp())
d = top/'course'; (d/'data').mkdir(parents=True); (d/'.git').mkdir()
two_notebooks(d)
(d/'notes.md').write_text('# Notes\n\nThe UNet predicts the noise.')
(d/'data/big.txt').write_text('data'); (d/'.git/config.txt').write_text('git'); (d/'model.pkl').write_text('bytes')
(top/'secret.txt').write_text('outside'); (d/'link.txt').symlink_to(top/'secret.txt'); os.link(top/'secret.txt', d/'key.txt')
(d/'tool.py').write_text('print(1)')

`tree_files` keeps the two notebooks and the note. It skips the data folder, the `.git` folder, the `.pkl` file, the link, the hard link `key.txt` and, since the folder has notebooks, `tool.py`.

In [ ]:
test_eq([p.name for p in tree_files(d)], ['01_a.ipynb', '02_b.ipynb', 'notes.md'])
test_eq(tree_files(d).cut, '')

The caps: with a cap of 2 files, or of 1 MB in all, `tree_files` keeps the first files and says why it stopped. A file over `TREE_MB` is never read.

In [ ]:
was = TREE_FILES, TREE_MB, TREE_TOTAL_MB
TREE_FILES = 2
t = tree_files(d); test_eq((len(t), t.cut), (2, 'abhyasa stopped at 2 files.'))
TREE_FILES, TREE_TOTAL_MB = 400, 1
(d/'big.md').write_bytes(b'x' * (2**20 - 10))
t = tree_files(d); test_eq((t.cut, 'big.md' in [p.name for p in t]), ('abhyasa stopped at 1 MB.', False))
TREE_TOTAL_MB, TREE_MB = 200, 0.5
test_eq(('big.md' in [p.name for p in tree_files(d)], tree_files(d).cut), (False, ''))
TREE_FILES, TREE_MB, TREE_TOTAL_MB = was; (d/'big.md').unlink()

### Which folders abhyasa files

`folder_why` says why abhyasa does not file a folder, or gives `''`. The folder must be a real folder in your home folder.
It must not be the home folder itself, a hidden folder (a name that starts with a dot) or a link. The test folder is not in
the home folder, so it is refused.

In [ ]:
#| export
def folder_why(p) -> str:
    "Why abhyasa does not file the folder `p` (the rules of `settings.source_root_why`), or '' when it can."
    return source_root_why(p) or ('' if Path(str(p)).expanduser().resolve().is_dir() else NO_FOLDER)

In [ ]:
test_eq(folder_why(d), 'abhyasa reads only folders in your home folder.')
test_eq(folder_why('~'), 'Name a folder in your home folder, not the home folder itself.')
test_eq(folder_why('~/.ssh'), 'abhyasa does not read a hidden folder or a link.')
test_eq(folder_why('~/no-such-folder-for-abhyasa'), NO_FOLDER)

### Filing a folder

`_file_tree` files the files of a folder on a shelf. A notebook or a file with the same mtime as its copy on the shelf is not
read again. Each document gets `meta.root`, `meta.path` and `meta.mtime`, and `meta.url` when `url` gives one. The result
has the shape of `file_link`, with `mtime`. Its `added` holds only the documents that were not on the shelf before, so an
Undo never forgets a document that you had before.

`file_folder` checks the folder with `folder_why` first. It raises `LinkError` with the plain reason. A document that was on the shelf before, filed by another tool such as `abhyasa-index`, keeps its meta. `cut` says when a cap stopped the filing.

In [ ]:
#| export
def _file_tree(lr:Learning, root, shelf:str, name:str, url=None) -> L:
    "File the files of the folder `root` on `shelf`, with `meta.root=name` and `meta.url=url(path in root)`. The docs, and in `added` the new ones."
    root, sh = Path(root).resolve(), lr.vault.shelf(shelf)
    files = tree_files(root)
    key = lambda d: d['meta'].get('path') or d['source']
    had = {key(d): d for d in sh.sources()}
    if nbs := [p for p in files if p.suffix.lower() == '.ipynb']: index_notebooks(lr.vault, (), shelf, files=nbs)
    for p in files:
        if p.suffix.lower() == '.ipynb' or ((h := had.get(str(p))) and h['meta'].get('mtime') == p.stat().st_mtime): continue
        try: sh.add_file(str(p), title=f'{p.parent.name}/{p.stem}')
        except Exception: continue   # a file that the vault cannot read is left out
    by, docs, added = {key(d): d for d in sh.sources()}, L(), L()
    for p in files:
        if not (d := by.get(str(p))): continue
        m = dict(root=name, path=str(p), mtime=p.stat().st_mtime, **({'url': url(p.relative_to(root).as_posix())} if url else {}))
        mine = str(p) not in had or had[str(p)]['meta'].get('root') == name   # a document that was there before, filed by another tool, keeps its meta
        if mine and any(d['meta'].get(k) != v for k, v in m.items()): sh.set_meta(d['id'], **m)
        docs.append(x := AttrDict(shelf=shelf, doc_id=d['id'], title=d['title'], url=m.get('url', str(p)), mtime=m['mtime']))
        if str(p) not in had: added.append(x)
    docs.added, docs.cut = added, files.cut
    return docs

def file_folder(lr:Learning, path, shelf:str='notebooks') -> L:
    "File the folder `path` in your home folder on `shelf`. Each doc keeps `meta.root` (the folder name) and `meta.path`. Raises `LinkError` for a folder that abhyasa does not file."
    if why := folder_why(path): raise LinkError(why)
    r = Path(path).expanduser().resolve()
    return _file_tree(lr, r, shelf, r.name)

The test folder files three documents with `meta.root`. A second filing reads nothing again and adds nothing.

In [ ]:
lr = temp_learning()
docs = _file_tree(lr, d, 'notebooks', 'course')
test_eq(len(docs), 3)
test_eq({lr.vault.shelf('notebooks').doc(x.doc_id)['meta']['root'] for x in docs}, {'course'})
test_eq(len(docs.added), 3)
again = _file_tree(lr, d, 'notebooks', 'course')
test_eq((again, len(again.added)), (docs, 0))
test_fail(lambda: file_folder(lr, d), contains='home folder')

### The roots in the vault

`vault_roots` gives each folder or repo on a shelf, with its count of documents. The Start box uses it to say
"fastbook · 20 notebooks · in your vault". `root_docs` gives the documents of one root, in path order.

In [ ]:
#| export
def vault_roots(lr:Learning, shelf:str='notebooks') -> dict:
    "Each folder or repo filed on `shelf` (its `meta.root`), to its count of documents."
    sh = lr.vault.shelf(shelf)
    q = f"select json_extract(meta, '$.root') as r, count(*) from [{sh.t.docs.name}] where r is not null group by r"
    return dict(sh.db.execute(q).fetchall())

def root_docs(lr:Learning, root:str, shelf:str='notebooks') -> L:
    "The documents of the folder or repo `root` on `shelf`, in path order, in the shape of `file_link`."
    sh = lr.vault.shelf(shelf)
    rows = sh.t.docs(where="json_extract(meta, '$.root') = ?", where_args=[root])
    ds = sorted(((json.loads(r['meta'] or '{}'), r) for r in rows), key=lambda x: str(x[0].get('path') or ''))
    return L(AttrDict(shelf=shelf, doc_id=r['id'], title=r['title'], url=m.get('url') or m.get('path') or r['source'], mtime=m.get('mtime')) for m, r in ds)

In [ ]:
test_eq(vault_roots(lr), {'course': 3})
test_eq(root_docs(lr, 'course').attrgot('title'), ['course/01_a', 'course/02_b', 'course/notes'])

A notebook on the `notebooks` shelf now opens by its title, with no shelf given.

In [ ]:
test_eq(doc_pages(lr, 'course/02_b').title, 'course/02_b')

### Undo

`unfile` forgets the documents of one filing. It forgets a document only when it is still the same file: the same id and
the same mtime. It gives the count.

In [ ]:
#| export
def unfile(lr:Learning, docs) -> int:
    "Forget the docs that one filing added, when each is still the same file (same id, same mtime). Returns the count."
    n = 0
    for x in map(AttrDict, docs):
        sh = lr.vault if x.shelf == 'store' else lr.vault.shelf(x.shelf)
        r = first(sh.t.docs(where='id=?', where_args=[x.doc_id]))
        if r and json.loads(r['meta'] or '{}').get('mtime') == x.get('mtime'): sh.forget(x.doc_id); n += 1
    return n

In [ ]:
test_eq(unfile(lr, docs.added[:1]), 1)
test_eq(vault_roots(lr), {'course': 2})
test_eq(unfile(lr, docs.added[:1]), 0)   # it is gone already

### A GitHub repo

`gh_name` reads `owner/name` from a GitHub link, or from the bare `owner/name`. A pair of common words, such as `and/or`,
is not a repo. A host that is not github.com gives None.

In [ ]:
#| export
GH_NAME = re.compile(r'([A-Za-z0-9][A-Za-z0-9-]{0,38})/([\w.-]{1,100})')
NOT_REPOS = frozenset('and/or either/or he/she his/her yes/no on/off input/output read/write true/false pros/cons i/o w/o n/a tcp/ip client/server'.split())

def gh_name(s:str) -> str|None:
    "`owner/name` of the GitHub repo that `s` names (a github.com link, or `owner/name`), or None."
    s = str(s or '').strip().rstrip('.,;:)')
    m = re.fullmatch(r'(?:https?://)?(?:www\.)?github\.com/([^/\s]+)/([^/\s#?]+)(?:[/#?]\S*)?', s, re.I)
    o, n = (m[1], m[2].removesuffix('.git')) if m else (s.split('/', 1) + [''])[:2]
    if not GH_NAME.fullmatch(f'{o}/{n}') or s.lower() in NOT_REPOS or not re.search('[A-Za-z]', o) or n.strip('.') == '': return None
    return f'{o}/{n}'

In [ ]:
test_eq(gh_name('fastai/diffusion-nbs'), 'fastai/diffusion-nbs')
test_eq(gh_name('https://github.com/fastai/diffusion-nbs.git'), 'fastai/diffusion-nbs')
test_eq(gh_name('github.com/fastai/course22/blob/master/README.md'), 'fastai/course22')
for s in ('and/or', '10/30', 'a/..', 'https://gitlab.com/a/b', '~/code/x', 'docs.python.org/3'): test_eq(gh_name(s), None)

`file_repo` clones only a public repo on github.com. It asks GitHub first (`gh_public`): a private repo, a repo that is
not there, or a repo bigger than `REPO_MB` MB is refused. Then `clone_repo` gives the HTTPS link that was checked to
fossick's `gh_clone`, which clones it into fossick's cache folder. It uses HTTPS only, never SSH, clones only the last
commit (depth 1), puts `--` before the link, and stops git after 600 seconds. It removes a clone that stopped half way,
and it changes no environment variable. abhyasa has no clone code of its own. fossick is a core dependency, so no extra
is necessary. A repo that is in the cache already is brought up to date, not cloned again. The research agent uses
`clone_repo` too, so a repo is on the disk one time. Each document keeps `meta.root='owner/name'` and `meta.url`, the GitHub page of the file.

In [ ]:
#| export
REPO_MB = 500   # the biggest repo that abhyasa clones, in MB
def gh_public(owner:str, name:str) -> dict|None:
    "What GitHub says of the repo `owner/name` (`private`, `size` in KB), or None when it has no such repo. With gheasy installed, its `gh_token` gives a token, for more calls an hour."
    from fossick import http_get
    try: from gheasy.core import gh_token; tok = gh_token()
    except ImportError: tok = None
    url, kw = f'https://api.github.com/repos/{owner}/{name}', dict(timeout=15, allow_private=False)
    r = http_get(url, **kw, **({'headers': {'Authorization': f'Bearer {tok}'}} if tok else {}))
    if tok and r.status == 401: r = http_get(url, **kw)   # a bad token: ask again with no token
    return json.loads(r.body) if r.status == 200 else None

def clone_repo(url:str, timeout:float=600) -> tuple:
    "`(owner/name, folder)` for the public GitHub repo `url` (a link or `owner/name`), cloned or brought up to date by fossick's `gh_clone` in its cache. Raises `LinkError` with a plain reason."
    if not (g := gh_name(url)): raise LinkError('abhyasa clones only a public repo on github.com.')
    info = gh_public(*g.split('/'))
    if not info or info.get('private'): raise LinkError(f'{g} is not a public repo on github.com.')
    if (info.get('size') or 0) > REPO_MB * 1024: raise LinkError(f'{g} is too big. abhyasa clones a repo of at most {REPO_MB} MB.')
    from fossick import gh_clone   # here, so that the import of this module stays light
    try: return g, Path(gh_clone(f'https://github.com/{g}', timeout=timeout)).resolve()
    except Exception as e: raise LinkError(f'abhyasa could not clone {g}.') from e

def file_repo(lr:Learning, url:str, shelf:str='notebooks') -> L:
    "Clone the public GitHub repo `url` (a link or `owner/name`) with `clone_repo`, then file it on `shelf`. Raises `LinkError` with a plain reason."
    g, d = clone_repo(url)
    return _file_tree(lr, d, shelf, g, url=lambda rel: f'https://github.com/{g}/blob/HEAD/{quote(rel)}')

`gh_public` sends a token from gheasy's `gh_token` when gheasy is installed. GitHub gives 60 calls an hour with no token. A bad token (401) gets one more call with no token, and with no gheasy no token goes. A fake `http_get` stands for GitHub.

In [ ]:
import fossick, sys, types

In [ ]:
sent = []
def fake_get(url, **kw):
    "GitHub: 401 for a bad token, else a public repo of 10 KB."
    sent.append((kw.get('headers') or {}).get('Authorization'))
    return AttrDict(status=401 if sent[-1] == 'Bearer bad' else 200, body=b'{"private": false, "size": 10}')
was_get, mods = fossick.http_get, {k: sys.modules.get(k) for k in ('gheasy', 'gheasy.core')}
def token(t): sys.modules['gheasy'] = sys.modules['gheasy.core'] = types.SimpleNamespace(gh_token=lambda: t)
fossick.http_get = fake_get
try:
    token('good'); test_eq(gh_public('a', 'b')['size'], 10); test_eq(sent, ['Bearer good'])
    sent.clear(); token('bad'); test_eq(gh_public('a', 'b')['size'], 10); test_eq(sent, ['Bearer bad', None])   # a bad token: one more call with none
    sent.clear(); sys.modules['gheasy'] = sys.modules['gheasy.core'] = None; gh_public('a', 'b'); test_eq(sent, [None])   # no gheasy: no token
finally:
    fossick.http_get = was_get
    for k, v in mods.items():
        if v is None: sys.modules.pop(k, None)
        else: sys.modules[k] = v

A test clone: a fake fossick `gh_clone` copies the test folder, and a fake `gh_public` says that the repo is public. Nothing goes to the network.

In [ ]:
import sys, types

In [ ]:
cache = Path(tempfile.mkdtemp())
def fake_clone(url, timeout=None):
    "A clone with no network: it copies the test folder the first time, and keeps the link and the time limit that it got."
    got.append((url, timeout)); to = cache/url.rsplit('/', 1)[-1]
    if not to.exists(): shutil.copytree(d, to)
    return to

got, real, real_clone = [], gh_public, fossick.gh_clone
fossick.gh_clone, gh_public = fake_clone, lambda o, n: dict(private=False, size=10)
try:
    lr = temp_learning()
    docs = file_repo(lr, 'https://github.com/fastai/course')
    m = lr.vault.shelf('notebooks').doc(docs[0].doc_id)['meta']
    test_eq((m['root'], m['url'], got), ('fastai/course', 'https://github.com/fastai/course/blob/HEAD/01_a.ipynb', [('https://github.com/fastai/course', 600)]))
    test_eq(file_repo(lr, 'fastai/course').attrgot('doc_id'), docs.attrgot('doc_id'))   # in the cache: the same documents
    gh_public = lambda o, n: dict(private=True); got.clear()
    test_fail(lambda: file_repo(lr, 'fastai/other'), contains='not a public repo'); test_eq(got, [])   # a private repo is never cloned
    gh_public = lambda o, n: dict(private=False, size=REPO_MB * 1024 + 1)
    test_fail(lambda: file_repo(lr, 'fastai/big'), contains='too big'); test_eq(got, [])
    test_fail(lambda: file_repo(lr, 'and/or'), contains='only a public repo')
finally: gh_public, fossick.gh_clone = real, real_clone

## The source of a passage

The source line uses `datetime` for the day that the tutor read a web page.

In [ ]:
#| export
from datetime import datetime

`source_of` gives the source line of a passage: one label, the title, a link to the document, the place and the date. A passage from the web has the label `from the web`. A passage from a document of yours has `your source`. Text with no passage is the tutor's own words. The walk, the talk and the recap file all draw this one line.

In [ ]:
#| export
WEB, YOURS, TUTOR, YOU = 'from the web', 'your source', 'tutor', 'you'   # the four labels of a text: decision 6 of 0.0.13

def _place(r) -> str:
    "Where the passage `r` is in its document: `section 3.2.1 · p. 4`. A section name with no number stays whole."
    sec = ' '.join(str(r.get('section') or '').split())
    if sec[:1].isdigit(): sec = f'section {sec.split()[0]}'
    return ' · '.join(x for x in (sec, r.get('page') and f"p. {r['page']}") if x)

def _day(t) -> str:
    "The time `t` as a day, such as `29 Oct 2026`; '' with no time."
    try: d = datetime.fromtimestamp(float(t))
    except (TypeError, ValueError, OverflowError, OSError): return ''
    return f'{d.day} {d:%b %Y}'

def _txt1(x) -> str: return ' '.join(x.split())[:80] if isinstance(x, str) else ''

def source_of(read:dict, j:dict|None=None) -> AttrDict:
    """The source line of the passage `read` (a step `read`, a slide source or a cite) in walk `j`: its `label`, its `title`,
    `href` (its document in the vault), `place` (the section and the page, or the site of a web page), `url` (a web page), `date`,
    and `episode` (the id of a Ramayana episode, for its link to vedicreader). The place starts with `what`, the kind of text,
    when the passage has one (a Ramayana episode is `Episode summary (vedicreader)`), so a summary never reads as a quote."""
    r, s = read or {}, str((read or {}).get('src') or '')
    shelf, doc = r.get('shelf'), r.get('doc_id')
    if not (shelf and doc) and s.startswith('vault:'): shelf, doc = s[6:], str(r.get('ref') or '').split('#')[0]
    url = r.get('url') if str(r.get('url') or '').startswith(('http://', 'https://')) else ''
    web = s == 'web'
    label = WEB if web else YOURS if r.get('text') or s else TUTOR   # a cite names where it is from, so it is a source too
    title = r.get('title') or ((j or {}).get('source') or {}).get('title') or r.get('site') or 'your text'
    other = '' if web or s.startswith('vault:') or s in ('text', 'doc') else s   # a source that is not in the vault, such as `notes`
    return AttrDict(label=label, title=str(title), href=f'/v/{shelf}/{doc}' if shelf and doc else '', place=' · '.join(x for x in (_txt1(r.get('what')), _place(r)) if x) or (r.get('site') or '' if web else other),
                    url=url, date=_day(r.get('at')) if web else '', episode=str(r.get('episode') or '') if s == 'vedicstory' else '')

A passage of a paper in the vault names the paper, the section number and the page, and links to the document.

In [ ]:
p = source_of(dict(shelf='store', doc_id='d1', title='Attention Is All You Need', section='3.2.1 Scaled Dot-Product Attention', page=4, text='We suspect.'))
test_eq((p.label, p.title, p.href, p.place, p.url, p.date), ('your source', 'Attention Is All You Need', '/v/store/d1', 'section 3.2.1 · p. 4', '', ''))

A passage from the web names its site and the day it was read, and keeps the link to the page.

In [ ]:
w = source_of(dict(src='web', shelf='store', doc_id='d2', title='Softmax explained', site='example.org', url='https://example.org/s', text='Q.', at=datetime(2026, 10, 29, 12).timestamp()))
test_eq((w.label, w.href, w.place, w.url, w.date), ('from the web', '/v/store/d2', 'example.org', 'https://example.org/s', '29 Oct 2026'))

A cite on a vault shelf links to its document. A step with no passage is the tutor's words. A link that is not a web page is not kept.

In [ ]:
test_eq(source_of(dict(src='vault:notes', ref='n1#2', title='My notes')).href, '/v/notes/n1')
test_eq(source_of(dict(src='vedicreader', ref='a:1', title='Ashtadhyayi 1 2')).place, 'vedicreader')
test_eq(source_of(dict(text=''), dict(source=dict(title='A paper'))).label, 'tutor')
test_eq(source_of(dict(src='web', url='javascript:alert(1)', text='Q.')).url, '')

A Ramayana episode is your source. Its title is the episode name, and its place is the kāṇḍa, the sarga and the verses of its Vālmīki cite. It keeps its episode id for the link to vedicreader. Its file path is not a web page, so it is not kept as `url`. With `what`, the place starts with the kind of text: an episode is a summary, not a quote of the poem.

In [ ]:
e = dict(src='vedicstory', ref='VR 1.1.1-5', title='Vālmīki questions Nārada', section='Bāla, sarga 1, verses 1-5', episode='valmiki_questions_narada', url='/x/episodes.json', text='Q.')
x = source_of(e)
test_eq((x.label, x.title, x.place, x.url, x.href, x.episode), ('your source', 'Vālmīki questions Nārada', 'Bāla, sarga 1, verses 1-5', '', '', 'valmiki_questions_narada'))
test_eq(source_of(dict(src='vault:notes', ref='n1', episode='x')).episode, '')
test_eq(source_of({**e, 'section': 'Bāla, sarga 1, verses 1-5', 'what': 'Episode summary (vedicreader)'}).place, 'Episode summary (vedicreader) · Bāla, sarga 1, verses 1-5')

## vedicreader

The import reads the files of the vedicreader repo. It does not read the website. `$VEDICREADER_ROOT` gives the path of the repo.

| source | concepts |
|---|---|
| Sanskrit lessons | One concept for each lesson. The lesson before it in the course is its prereq. Its `check` pairs become ready questions at depth 4 (apply). Its recap becomes the statement. |
| `static/grammar/terms/*.json` | One concept for each term, with `related` links. A term with a jyotiṣa term of the same name also links to it. |
| `vr/jyotisha/explain/*.json` | One concept for each graha, rāśi and nakṣatra. |

`VR_ROOT` is the repo path. `plain` removes the marks of vedicreader from a text.

In [ ]:
#| export
VR_ROOT = Path(os.getenv('VEDICREADER_ROOT', Path.home()/'code/personal/orgs/vedicreader'))

def plain(s:str) -> str:
    "The vedicreader text without its `{…}` marks and `[1,2]` citations."
    return re.sub(r'\[\d+(?:,\s*\d+)*\]', '', re.sub(r'\{([^{}]*)\}', r'\1', s or '')).strip()

Three generators read one source each. Each generator yields the concepts as dicts for `put`. `_lesson_concepts` reads the lessons.

In [ ]:
#| export
def _lesson_concepts(root):
    d = root/'static/learn'
    order, prev = json.loads((d/'order.json').read_text()), None
    for ui, u in enumerate(order['units']):
        for li, ls in enumerate(u['lessons']):
            p = d/'lessons'/f"{ls['slug']}.json"
            body = json.loads(p.read_text()).get('body', []) if p.exists() else []
            recap = [plain(x) for b in body for x in b.get('recap', [])]
            qs = [dict(q=plain(c['q']), a=plain(c['a']), depth=4) for b in body for c in b.get('check', [])]
            yield dict(name=ls['title'], statement=' '.join([plain(ls['cap']), *recap]), subject='sanskrit', slug=f"sa-{ls['slug']}",
                       prereqs=[prev] if prev else [], questions=qs, order=ui*100 + li, unit=u['name'],
                       provenance=[dict(src='vedicreader', lesson=ls['slug'], url=f"https://vedicreader.com/v/learn/{ls['slug']}")])
            prev = f"sa-{ls['slug']}"

`_term_concepts` reads the grammar terms.

In [ ]:
#| export
def _term_concepts(root):
    for p in sorted((root/'static/grammar/terms').glob('*.json')):
        for t in json.loads(p.read_text()).get('terms', []):
            en = ', '.join(t.get('en') or [])
            yield dict(name=t.get('iast') or t['slug'], statement=plain(t.get('defn', '')), subject='sanskrit', slug=f"sa-term-{t['slug']}",
                       aliases=[a for a in (t.get('sa'), *(t.get('en') or ())) if a], group=t.get('group', ''), order=1000,
                       related=[f"sa-term-{r}" for r in t.get('related') or ()] + ([f"jy-term-{t['jy']}"] if t.get('jy') else []),
                       provenance=[dict(src='vedicreader', term=t['slug'], cite=(t.get('src') or {}).get('cite', ''))],
                       questions=[dict(q=f"What is {t.get('iast') or t['slug']} ({en})? Say what it is and where it is used.",
                                       a=plain(t.get('defn', '')), depth=2)] if t.get('defn') else [])

`_explain_concepts` reads the jyotiṣa explanations. The rāśi file keys each sign by its number, so `RASI` gives the names.

In [ ]:
#| export
RASI = [('Meṣa', 'Aries'), ('Vṛṣabha', 'Taurus'), ('Mithuna', 'Gemini'), ('Karkaṭa', 'Cancer'), ('Siṃha', 'Leo'), ('Kanyā', 'Virgo'),
        ('Tulā', 'Libra'), ('Vṛścika', 'Scorpio'), ('Dhanus', 'Sagittarius'), ('Makara', 'Capricorn'), ('Kumbha', 'Aquarius'), ('Mīna', 'Pisces')]

def _explain_name(kind, key):
    "The name and aliases of one explained item; the rāśi file keys each sign by its number, 0 for Meṣa."
    if kind == 'rasi' and key.isdigit(): nm, en = RASI[int(key)]; return nm, [en, 'rāśi', 'sign']
    return key, [kind]

def _explain_concepts(root):
    for kind in ('graha', 'rasi', 'nakshatra'):
        p = root/f'vr/jyotisha/explain/{kind}.json'
        if not p.exists(): continue
        for name, v in json.loads(p.read_text()).items():
            secs = v.get('sections', {})
            first_sec = next(iter(secs.values()), [])
            st = plain(first_sec[0] if first_sec else '')[:900]
            nm, al = _explain_name(kind, name)
            yield dict(name=nm, statement=st, subject='jyotisha', slug=f"jy-{kind}-{slugify(nm)}", kind=kind, order=2000,
                       aliases=al, provenance=[dict(src='vedicreader', explain=kind, key=name)])

`from_vedicreader` runs the generators that `parts` names and returns a count for each part.

In [ ]:
#| export
def from_vedicreader(lr:Learning, root=None, parts=('lessons', 'terms', 'explain')) -> dict:
    "Add the lessons, terms and jyotiṣa explanations of vedicreader as concepts. Keep the learner state on the shelf."
    root, n = Path(root or VR_ROOT), {}
    gens = dict(lessons=_lesson_concepts, terms=_term_concepts, explain=_explain_concepts)
    for part in parts:
        n[part] = 0
        for c in gens[part](root):
            lr.put(**c); n[part] += 1
    return n

`plain` removes the marks and the citations.

In [ ]:
test_eq(plain('{a} + {a} → {ā}: {nāsti}.[1,2]'), 'a + a → ā: nāsti.')

When the repo is on this machine, the lessons become a chain of concepts with prereqs and ready questions.

In [ ]:
if VR_ROOT.exists():
    lr = temp_learning()
    n = from_vedicreader(lr, parts=('lessons',))
    assert n['lessons'] > 10
    c = lr.concept('sa-sandhi-vowels')
    assert c.prereqs and c.questions and c.subject == 'sanskrit'
    test_eq(lr.concept('sa-sounds').prereqs, [])
    test_eq(_explain_name('rasi', '0'), ('Meṣa', ['Aries', 'rāśi', 'sign']))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()